# AMR Sentinel — WHO E. coli CSV Upload, EDA & Machine Learning (Colab)

**Run in Google Colab (CPU runtime).** Upload **one real WHO E. coli resistance CSV** when prompted. No WHO network API calls, `venv`, `ensurepip` or Google Drive mount are required. The notebook then validates the CSV, creates national and WHO-region descriptive statistics, runs EDA, trains and evaluates forecasting models (naïve, linear, Ridge), creates an HTML report, and downloads a ZIP.

**Source:** [WHO E. coli SDG 3.d.2 indicator](https://data.who.int/indicators/i/918081E/745F475). Input must represent *E. coli* third-generation-cephalosporin resistance in bloodstream infections (percentage, **0–100**, not a fraction). If provenance cannot be established, do not label the file WHO-derived.

**Scope note:** Regional percentages are **unweighted medians of reporting countries, calculated here**. They are not WHO regional adjusted estimates, patient prevalence, clinical guidance, real-time warnings or 2026 surveillance results. The latest imported observed year determines the historical holdout and experimental one-year-ahead forecast.

**Run order:** (1) Colab check → (2) dependencies → (3) upload CSV → (4) settings → (5) define validation/EDA/model stages → (6) Run full study → (7) review EDA & report → (8) download bundle.


In [ ]:
import sys, os, json, subprocess
from pathlib import Path
try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Run this end-to-end analysis only in Google Colab; no local model training is authorized.") from exc
if "google.colab" not in sys.modules or not Path("/content").is_dir():
    raise RuntimeError("A Google Colab runtime is required.")
print("Google Colab confirmed. CPU is sufficient; no GPU or Drive mount is needed.")

## 1. Colab Python environment

This notebook uses Colab's native Python; it never creates a virtual environment. Existing scientific libraries are checked in a fresh worker process. If needed, compatible binary packages are installed to an isolated target directory. Package errors are recorded in `/content/amr_sentinel/environment_install.log`.


In [ ]:
# Google Colab compatible setup. No virtualenv, ensurepip, or in-kernel scientific imports.
import sys, os, json, subprocess, shutil
from pathlib import Path

if not ((3, 11) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(f"Supported Google Colab Python versions: 3.11–3.13; found {sys.version.split()[0]}.")

WORK_ROOT = Path("/content/amr_sentinel")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
WORK = WORK_ROOT  # Used by the settings cell; replaced with a unique run folder for each analysis.
worker_python = sys.executable  # The already-working Colab Python interpreter.
install_log = WORK_ROOT / "environment_install.log"
dependency_dir = Path("/content/amr_sentinel_dependencies")
dependency_stamp = dependency_dir / ".amr_ready.json"

# Required modules only. PyArrow is optional; the analysis always writes CSV.
packages = [
    "numpy>=2.2,<3", "scipy>=1.15,<2", "pandas>=2.2,<3",
    "scikit-learn>=1.6,<2", "matplotlib>=3.9,<4",
    "requests>=2.32,<3", "joblib>=1.4,<2",
]
probe = (
    "import numpy, scipy, pandas, sklearn, matplotlib, requests, joblib; "
    "from sklearn.linear_model import Ridge; "
    "print('Scientific imports OK | Python:', __import__('sys').version.split()[0], "
    "'| numpy:', numpy.__version__, '| scipy:', scipy.__version__, "
    "'| pandas:', pandas.__version__, '| sklearn:', sklearn.__version__)"
)

# Keep the notebook kernel and the worker separate. No dependency imports above.
worker_env = os.environ.copy()
worker_env.pop("PYTHONPATH", None)
worker_env["PYTHONNOUSERSITE"] = "1"

def checked(command, label, env=None):
    completed = subprocess.run(
        [str(v) for v in command], capture_output=True, text=True, env=env,
    )
    combined = (completed.stdout or "") + (("\n" + completed.stderr) if completed.stderr else "")
    with install_log.open("a", encoding="utf-8") as logfile:
        logfile.write(f"\n--- {label} ---\n$ {' '.join(map(str, command))}\n")
        logfile.write(combined or "(no output)\n")
    if completed.returncode:
        print(f"{label} FAILED (exit {completed.returncode})")
        print(combined[-10000:] if combined.strip() else "No subprocess diagnostics were returned.")
    else:
        print(f"{label}: OK")
        if label.startswith("Probe"):
            print(combined.strip())
    return completed

install_log.write_text("AMR Sentinel: Colab native Python setup (no virtualenv)\n", encoding="utf-8")
base_probe = checked([worker_python, "-c", probe], "Probe Colab preinstalled packages", worker_env)
if base_probe.returncode == 0:
    print("Using Colab's existing scientific packages; no installation needed.")
else:
    print("Colab package import check failed; preparing an isolated pip --target directory.")
    # Preserve a complete, previously validated directory if present.
    cache_valid = False
    if dependency_stamp.is_file():
        try:
            cache_valid = json.loads(dependency_stamp.read_text(encoding="utf-8")) == {
                "python": list(sys.version_info[:2]), "packages": packages
            }
        except (OSError, ValueError):
            pass
    if cache_valid:
        worker_env["PYTHONPATH"] = str(dependency_dir)
        cache_probe = checked([worker_python, "-c", probe], "Probe cached target packages", worker_env)
        cache_valid = cache_probe.returncode == 0
    if not cache_valid:
        # A different new directory prevents partially installed packages being reused.
        from uuid import uuid4
        temp_dir = Path("/content") / f"amr_deps_install_{uuid4().hex[:10]}"
        pip_version = checked([worker_python, "-m", "pip", "--version"], "Check Colab pip", worker_env)
        if pip_version.returncode:
            raise RuntimeError(
                "Colab's Python has no working pip. Use a standard Colab runtime or reset the runtime. "
                f"Full details: {install_log}"
            )
        installed = checked(
            [worker_python, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-cache-dir", "--only-binary=:all:", "--target", str(temp_dir)] + packages,
            "Install binary packages to isolated target directory", worker_env,
        )
        if installed.returncode:
            raise RuntimeError(f"Package installation failed; see {install_log} for the pip error.")
        target_env = worker_env.copy()
        target_env["PYTHONPATH"] = str(temp_dir)
        target_probe = checked([worker_python, "-c", probe], "Probe new target packages", target_env)
        if target_probe.returncode:
            raise RuntimeError(f"New target packages could not be imported; see {install_log}.")
        # Commit the completed installation only after successful import.
        if dependency_dir.exists():
            shutil.rmtree(dependency_dir)
        temp_dir.rename(dependency_dir)
        dependency_stamp.write_text(json.dumps({
            "python": list(sys.version_info[:2]), "packages": packages
        }, indent=2), encoding="utf-8")
        worker_env["PYTHONPATH"] = str(dependency_dir)
        verified = checked([worker_python, "-c", probe], "Probe committed target packages", worker_env)
        if verified.returncode:
            raise RuntimeError(f"Committed packages failed verification; see {install_log}.")

print("Analysis interpreter:", worker_python)
print("Extra packages:", worker_env.get("PYTHONPATH", "none (using standard Colab packages)"))
print("Environment log:", install_log)


## 2. Upload your one WHO E. coli CSV

Run the next cell and select a **single `.csv`** from your computer. File bytes are preserved and SHA-256 hashed. Accepted national fields include:

| Required information | Accepted examples |
|---|---|
| Country code or name | `DIM_GEO_CODE_ISO3`, `SpatialDim`, `country_code`, `country`, `DIM_GEO_NAME`, `DIM_GEO_CODE_M49` |
| Year | `DIM_TIME_YEAR`, `TimeDim`, `YEAR`, `year` |
| Resistance (%) | `AMOUNT_N`, `NumericValue`, `OBS_VALUE`, `resistance_pct`, `VALUE` |
| WHO region (optional) | `who_region`, `WHO_REGION`, `GRP_WHO_REGION` |

If WHO region is absent, a **bundled WHO Member State region lookup** (194 members; classification snapshot May 2025) is used offline. Unmapped countries are reported and excluded, not guessed. Historical country-region membership can differ (notably Indonesia before May 2025). If your CSV uses unusual column names, the validation error will show its available columns; rename them in a copy and re-upload. This is a national-file workflow, **not** a direct WHO regional aggregate importer.

The upload cell prints the original headers and SHA-256; the EDA output `data/processed/input_validation.json` documents chosen fields, missing percentages and unmapped countries. Data are **not** asserted WHO-authentic until independently verified against the original source.


In [ ]:
# Choose exactly one local WHO E. coli CSV; no server download.
from pathlib import Path
import hashlib, csv, io, json
from google.colab import files
UPLOADED = files.upload()  # A browser file dialog appears in Colab.
if len(UPLOADED) != 1:
    raise ValueError(f"Upload exactly one .csv file; received {len(UPLOADED)}.")
original_filename, raw_bytes = next(iter(UPLOADED.items()))
if Path(original_filename).suffix.lower() != ".csv":
    raise ValueError("Only a single WHO CSV file is accepted.")
if not raw_bytes or len(raw_bytes) > 100_000_000:
    raise ValueError("CSV is empty or larger than 100 MB.")
if raw_bytes.lstrip().startswith((b"<!DOCTYPE", b"<html")):
    raise ValueError("Uploaded content appears to be HTML, not a CSV.")
INPUT_DIR = WORK_ROOT / "uploaded_input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_CSV = INPUT_DIR / "who_ecoli_uploaded.csv"
INPUT_CSV.write_bytes(raw_bytes)
INPUT_META = {
    "original_filename": Path(original_filename).name,
    "sha256": hashlib.sha256(raw_bytes).hexdigest(),
    "size_bytes": len(raw_bytes),
    "uploaded_utc": __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
    "claimed_source_url": "https://data.who.int/indicators/i/918081E/745F475",
    "source_status": "user-supplied; WHO provenance should be independently verified",
}
(INPUT_DIR / "upload_metadata.json").write_text(json.dumps(INPUT_META, indent=2), encoding="utf-8")
header = next(csv.reader(io.StringIO(raw_bytes.decode('utf-8-sig',errors='replace'))), [])
print('CSV uploaded:', original_filename)
print('Size:', len(raw_bytes), 'bytes | SHA-256:', INPUT_META['sha256'])
print('Columns:', header)
print('Stored locally:', INPUT_CSV)


## 3. Study configuration

Country-to-region aggregation requires at least five reporting countries in a given region/year. Ineligible groups are retained in coverage diagnostics but excluded from ML. The final holdout is the latest **observed** year. Only experimental next-year forecasts are emitted. Do not edit the target definition unless the uploaded CSV measures a different indicator and the entire project is redesigned.


In [ ]:
run_settings = {
    "holdout_year": None,
    "indicator": "ECOLI_3GC_BSI",
    "statistic": "derived_country_median",
    "minimum_reporting_countries": 5,
    "data_acquisition": "single_user_uploaded_csv_offline",
    "country_mapping_vintage": "WHO Member State lookup, May 2025 snapshot",
    "primary_who_source": "https://data.who.int/indicators/i/918081E/745F475",
}
print(json.dumps(run_settings, indent=2))


## 4. CSV validation, country-to-region mapping and data preparation

The code below is stored as source text and executed only by the Colab worker. It uses the uploaded CSV bytes, chooses documented national data fields and WHO membership mapping, validates 0–100 percentages, rejects conflicting duplicate country/year observations, and calculates **unweighted regional medians** with at least five reporting countries. It does **not** contact WHO servers. A source manifest preserves file hash, original schema, and the distinction between a user-uploaded file and independently verified WHO data.


In [ ]:
acquisition_source = r'''
# Local user-supplied CSV ingestion, no HTTP requests.
REGIONS = ("AFR", "AMR", "EMR", "EUR", "SEAR", "WPR")
INDICATOR_PAGE = "https://data.who.int/indicators/i/918081E/745F475"
WHO_MAPPING_SOURCE = "https://www.who.int/countries (May 2025 regional grouping; embedded reference table)"
WHO_MAPPING_SUPPORT = "https://shanlong-who.github.io/DSIR/reference/who_countries.html"
COUNTRY_TO_REGION = {'DZA': 'AFR', 'AGO': 'AFR', 'BEN': 'AFR', 'BWA': 'AFR', 'BFA': 'AFR', 'BDI': 'AFR', 'CPV': 'AFR', 'CMR': 'AFR', 'CAF': 'AFR', 'TCD': 'AFR', 'COM': 'AFR', 'COG': 'AFR', 'CIV': 'AFR', 'COD': 'AFR', 'GNQ': 'AFR', 'ERI': 'AFR', 'SWZ': 'AFR', 'ETH': 'AFR', 'GAB': 'AFR', 'GMB': 'AFR', 'GHA': 'AFR', 'GIN': 'AFR', 'GNB': 'AFR', 'KEN': 'AFR', 'LSO': 'AFR', 'LBR': 'AFR', 'MDG': 'AFR', 'MWI': 'AFR', 'MLI': 'AFR', 'MRT': 'AFR', 'MUS': 'AFR', 'MOZ': 'AFR', 'NAM': 'AFR', 'NER': 'AFR', 'NGA': 'AFR', 'RWA': 'AFR', 'STP': 'AFR', 'SEN': 'AFR', 'SYC': 'AFR', 'SLE': 'AFR', 'ZAF': 'AFR', 'SSD': 'AFR', 'TGO': 'AFR', 'UGA': 'AFR', 'TZA': 'AFR', 'ZMB': 'AFR', 'ZWE': 'AFR', 'ATG': 'AMR', 'ARG': 'AMR', 'BHS': 'AMR', 'BRB': 'AMR', 'BLZ': 'AMR', 'BOL': 'AMR', 'BRA': 'AMR', 'CAN': 'AMR', 'CHL': 'AMR', 'COL': 'AMR', 'CRI': 'AMR', 'CUB': 'AMR', 'DMA': 'AMR', 'DOM': 'AMR', 'ECU': 'AMR', 'SLV': 'AMR', 'GRD': 'AMR', 'GTM': 'AMR', 'GUY': 'AMR', 'HTI': 'AMR', 'HND': 'AMR', 'JAM': 'AMR', 'MEX': 'AMR', 'NIC': 'AMR', 'PAN': 'AMR', 'PRY': 'AMR', 'PER': 'AMR', 'KNA': 'AMR', 'LCA': 'AMR', 'VCT': 'AMR', 'SUR': 'AMR', 'TTO': 'AMR', 'USA': 'AMR', 'URY': 'AMR', 'VEN': 'AMR', 'AFG': 'EMR', 'BHR': 'EMR', 'DJI': 'EMR', 'EGY': 'EMR', 'IRN': 'EMR', 'IRQ': 'EMR', 'JOR': 'EMR', 'KWT': 'EMR', 'LBN': 'EMR', 'LBY': 'EMR', 'MAR': 'EMR', 'OMN': 'EMR', 'PAK': 'EMR', 'QAT': 'EMR', 'SAU': 'EMR', 'SOM': 'EMR', 'SDN': 'EMR', 'SYR': 'EMR', 'TUN': 'EMR', 'ARE': 'EMR', 'YEM': 'EMR', 'BGD': 'SEAR', 'BTN': 'SEAR', 'PRK': 'SEAR', 'IND': 'SEAR', 'MDV': 'SEAR', 'MMR': 'SEAR', 'NPL': 'SEAR', 'LKA': 'SEAR', 'THA': 'SEAR', 'TLS': 'SEAR', 'ALB': 'EUR', 'AND': 'EUR', 'ARM': 'EUR', 'AUT': 'EUR', 'AZE': 'EUR', 'BLR': 'EUR', 'BEL': 'EUR', 'BIH': 'EUR', 'BGR': 'EUR', 'HRV': 'EUR', 'CYP': 'EUR', 'CZE': 'EUR', 'DNK': 'EUR', 'EST': 'EUR', 'FIN': 'EUR', 'FRA': 'EUR', 'GEO': 'EUR', 'DEU': 'EUR', 'GRC': 'EUR', 'HUN': 'EUR', 'ISL': 'EUR', 'IRL': 'EUR', 'ISR': 'EUR', 'ITA': 'EUR', 'KAZ': 'EUR', 'KGZ': 'EUR', 'LVA': 'EUR', 'LTU': 'EUR', 'LUX': 'EUR', 'MLT': 'EUR', 'MCO': 'EUR', 'MNE': 'EUR', 'NLD': 'EUR', 'MKD': 'EUR', 'NOR': 'EUR', 'POL': 'EUR', 'PRT': 'EUR', 'MDA': 'EUR', 'ROU': 'EUR', 'RUS': 'EUR', 'SMR': 'EUR', 'SRB': 'EUR', 'SVK': 'EUR', 'SVN': 'EUR', 'ESP': 'EUR', 'SWE': 'EUR', 'CHE': 'EUR', 'TJK': 'EUR', 'TUR': 'EUR', 'TKM': 'EUR', 'UKR': 'EUR', 'GBR': 'EUR', 'UZB': 'EUR', 'AUS': 'WPR', 'BRN': 'WPR', 'KHM': 'WPR', 'CHN': 'WPR', 'COK': 'WPR', 'FJI': 'WPR', 'IDN': 'WPR', 'JPN': 'WPR', 'KIR': 'WPR', 'LAO': 'WPR', 'MYS': 'WPR', 'MHL': 'WPR', 'FSM': 'WPR', 'MNG': 'WPR', 'NRU': 'WPR', 'NZL': 'WPR', 'NIU': 'WPR', 'PLW': 'WPR', 'PNG': 'WPR', 'PHL': 'WPR', 'KOR': 'WPR', 'WSM': 'WPR', 'SGP': 'WPR', 'SLB': 'WPR', 'TON': 'WPR', 'TUV': 'WPR', 'VUT': 'WPR', 'VNM': 'WPR', 'PSE': 'EMR', 'PRI': 'AMR', 'TKL': 'WPR'}
COUNTRY_ALIASES = {'100': 'BGR', '104': 'MMR', '108': 'BDI', '112': 'BLR', '116': 'KHM', '12': 'DZA', '120': 'CMR', '124': 'CAN', '132': 'CPV', '140': 'CAF', '144': 'LKA', '148': 'TCD', '152': 'CHL', '156': 'CHN', '170': 'COL', '174': 'COM', '178': 'COG', '180': 'COD', '184': 'COK', '188': 'CRI', '191': 'HRV', '192': 'CUB', '196': 'CYP', '20': 'AND', '203': 'CZE', '204': 'BEN', '208': 'DNK', '212': 'DMA', '214': 'DOM', '218': 'ECU', '222': 'SLV', '226': 'GNQ', '231': 'ETH', '232': 'ERI', '233': 'EST', '24': 'AGO', '242': 'FJI', '246': 'FIN', '250': 'FRA', '262': 'DJI', '266': 'GAB', '268': 'GEO', '270': 'GMB', '275': 'PSE', '276': 'DEU', '28': 'ATG', '288': 'GHA', '296': 'KIR', '300': 'GRC', '308': 'GRD', '31': 'AZE', '32': 'ARG', '320': 'GTM', '324': 'GIN', '328': 'GUY', '332': 'HTI', '340': 'HND', '348': 'HUN', '352': 'ISL', '356': 'IND', '36': 'AUS', '360': 'IDN', '364': 'IRN', '368': 'IRQ', '372': 'IRL', '376': 'ISR', '380': 'ITA', '384': 'CIV', '388': 'JAM', '392': 'JPN', '398': 'KAZ', '4': 'AFG', '40': 'AUT', '400': 'JOR', '404': 'KEN', '408': 'PRK', '410': 'KOR', '414': 'KWT', '417': 'KGZ', '418': 'LAO', '422': 'LBN', '426': 'LSO', '428': 'LVA', '430': 'LBR', '434': 'LBY', '44': 'BHS', '440': 'LTU', '442': 'LUX', '450': 'MDG', '454': 'MWI', '458': 'MYS', '462': 'MDV', '466': 'MLI', '470': 'MLT', '478': 'MRT', '48': 'BHR', '480': 'MUS', '484': 'MEX', '492': 'MCO', '496': 'MNG', '498': 'MDA', '499': 'MNE', '50': 'BGD', '504': 'MAR', '508': 'MOZ', '51': 'ARM', '512': 'OMN', '516': 'NAM', '52': 'BRB', '520': 'NRU', '524': 'NPL', '528': 'NLD', '548': 'VUT', '554': 'NZL', '558': 'NIC', '56': 'BEL', '562': 'NER', '566': 'NGA', '570': 'NIU', '578': 'NOR', '583': 'FSM', '584': 'MHL', '585': 'PLW', '586': 'PAK', '591': 'PAN', '598': 'PNG', '600': 'PRY', '604': 'PER', '608': 'PHL', '616': 'POL', '620': 'PRT', '624': 'GNB', '626': 'TLS', '630': 'PRI', '634': 'QAT', '64': 'BTN', '642': 'ROU', '643': 'RUS', '646': 'RWA', '659': 'KNA', '662': 'LCA', '670': 'VCT', '674': 'SMR', '678': 'STP', '68': 'BOL', '682': 'SAU', '686': 'SEN', '688': 'SRB', '690': 'SYC', '694': 'SLE', '70': 'BIH', '702': 'SGP', '703': 'SVK', '704': 'VNM', '705': 'SVN', '706': 'SOM', '710': 'ZAF', '716': 'ZWE', '72': 'BWA', '724': 'ESP', '728': 'SSD', '729': 'SDN', '740': 'SUR', '748': 'SWZ', '752': 'SWE', '756': 'CHE', '76': 'BRA', '760': 'SYR', '762': 'TJK', '764': 'THA', '768': 'TGO', '772': 'TKL', '776': 'TON', '780': 'TTO', '784': 'ARE', '788': 'TUN', '792': 'TUR', '795': 'TKM', '798': 'TUV', '8': 'ALB', '800': 'UGA', '804': 'UKR', '807': 'MKD', '818': 'EGY', '826': 'GBR', '834': 'TZA', '84': 'BLZ', '840': 'USA', '854': 'BFA', '858': 'URY', '860': 'UZB', '862': 'VEN', '882': 'WSM', '887': 'YEM', '894': 'ZMB', '90': 'SLB', '96': 'BRN', 'AD': 'AND', 'AE': 'ARE', 'AF': 'AFG', 'AFGHANISTAN': 'AFG', 'AG': 'ATG', 'AL': 'ALB', 'ALBANIA': 'ALB', 'ALGERIA': 'DZA', 'AM': 'ARM', 'ANDORRA': 'AND', 'ANGOLA': 'AGO', 'ANTIGUA AND BARBUDA': 'ATG', 'AO': 'AGO', 'AR': 'ARG', 'ARAB REPUBLIC OF EGYPT': 'EGY', 'ARGENTINA': 'ARG', 'ARGENTINE REPUBLIC': 'ARG', 'ARMENIA': 'ARM', 'AT': 'AUT', 'AU': 'AUS', 'AUSTRALIA': 'AUS', 'AUSTRIA': 'AUT', 'AZ': 'AZE', 'AZERBAIJAN': 'AZE', 'BA': 'BIH', 'BAHAMAS': 'BHS', 'BAHRAIN': 'BHR', 'BANGLADESH': 'BGD', 'BARBADOS': 'BRB', 'BB': 'BRB', 'BD': 'BGD', 'BE': 'BEL', 'BELARUS': 'BLR', 'BELGIUM': 'BEL', 'BELIZE': 'BLZ', 'BENIN': 'BEN', 'BF': 'BFA', 'BG': 'BGR', 'BH': 'BHR', 'BHUTAN': 'BTN', 'BI': 'BDI', 'BJ': 'BEN', 'BN': 'BRN', 'BO': 'BOL', 'BOLIVARIAN REPUBLIC OF VENEZUELA': 'VEN', 'BOLIVIA': 'BOL', 'BOLIVIA (PLURINATIONAL STATE OF)': 'BOL', 'BOLIVIA, PLURINATIONAL STATE OF': 'BOL', 'BOSNIA AND HERZEGOVINA': 'BIH', 'BOTSWANA': 'BWA', 'BR': 'BRA', 'BRAZIL': 'BRA', 'BRUNEI': 'BRN', 'BRUNEI DARUSSALAM': 'BRN', 'BS': 'BHS', 'BT': 'BTN', 'BULGARIA': 'BGR', 'BURKINA FASO': 'BFA', 'BURUNDI': 'BDI', 'BW': 'BWA', 'BY': 'BLR', 'BZ': 'BLZ', 'CA': 'CAN', 'CABO VERDE': 'CPV', 'CAMBODIA': 'KHM', 'CAMEROON': 'CMR', 'CANADA': 'CAN', 'CAPE VERDE': 'CPV', 'CD': 'COD', 'CENTRAL AFRICAN REPUBLIC': 'CAF', 'CF': 'CAF', 'CG': 'COG', 'CH': 'CHE', 'CHAD': 'TCD', 'CHILE': 'CHL', 'CHINA': 'CHN', 'CI': 'CIV', 'CK': 'COK', 'CL': 'CHL', 'CM': 'CMR', 'CN': 'CHN', 'CO': 'COL', 'COLOMBIA': 'COL', 'COMMONWEALTH OF DOMINICA': 'DMA', 'COMMONWEALTH OF THE BAHAMAS': 'BHS', 'COMOROS': 'COM', 'CONGO': 'COG', 'CONGO, THE DEMOCRATIC REPUBLIC OF THE': 'COD', 'COOK ISLANDS': 'COK', 'COSTA RICA': 'CRI', "COTE D'IVOIRE": 'CIV', 'CR': 'CRI', 'CROATIA': 'HRV', 'CU': 'CUB', 'CUBA': 'CUB', 'CV': 'CPV', 'CY': 'CYP', 'CYPRUS': 'CYP', 'CZ': 'CZE', 'CZECH REPUBLIC': 'CZE', 'CZECHIA': 'CZE', "CÔTE D'IVOIRE": 'CIV', 'CÔTE D’IVOIRE': 'CIV', 'DE': 'DEU', "DEMOCRATIC PEOPLE'S REPUBLIC OF KOREA": 'PRK', 'DEMOCRATIC REPUBLIC OF SAO TOME AND PRINCIPE': 'STP', 'DEMOCRATIC REPUBLIC OF THE CONGO': 'COD', 'DEMOCRATIC REPUBLIC OF TIMOR-LESTE': 'TLS', 'DEMOCRATIC SOCIALIST REPUBLIC OF SRI LANKA': 'LKA', 'DENMARK': 'DNK', 'DJ': 'DJI', 'DJIBOUTI': 'DJI', 'DK': 'DNK', 'DM': 'DMA', 'DO': 'DOM', 'DOMINICA': 'DMA', 'DOMINICAN REPUBLIC': 'DOM', 'DR CONGO': 'COD', 'DZ': 'DZA', 'EAST TIMOR': 'TLS', 'EASTERN REPUBLIC OF URUGUAY': 'URY', 'EC': 'ECU', 'ECUADOR': 'ECU', 'EE': 'EST', 'EG': 'EGY', 'EGYPT': 'EGY', 'EL SALVADOR': 'SLV', 'EQUATORIAL GUINEA': 'GNQ', 'ER': 'ERI', 'ERITREA': 'ERI', 'ES': 'ESP', 'ESTONIA': 'EST', 'ESWATINI': 'SWZ', 'ET': 'ETH', 'ETHIOPIA': 'ETH', 'FEDERAL DEMOCRATIC REPUBLIC OF ETHIOPIA': 'ETH', 'FEDERAL DEMOCRATIC REPUBLIC OF NEPAL': 'NPL', 'FEDERAL REPUBLIC OF GERMANY': 'DEU', 'FEDERAL REPUBLIC OF NIGERIA': 'NGA', 'FEDERAL REPUBLIC OF SOMALIA': 'SOM', 'FEDERATED STATES OF MICRONESIA': 'FSM', 'FEDERATIVE REPUBLIC OF BRAZIL': 'BRA', 'FI': 'FIN', 'FIJI': 'FJI', 'FINLAND': 'FIN', 'FJ': 'FJI', 'FM': 'FSM', 'FR': 'FRA', 'FRANCE': 'FRA', 'FRENCH REPUBLIC': 'FRA', 'GA': 'GAB', 'GABON': 'GAB', 'GABONESE REPUBLIC': 'GAB', 'GAMBIA': 'GMB', 'GB': 'GBR', 'GD': 'GRD', 'GE': 'GEO', 'GEORGIA': 'GEO', 'GERMANY': 'DEU', 'GH': 'GHA', 'GHANA': 'GHA', 'GM': 'GMB', 'GN': 'GIN', 'GQ': 'GNQ', 'GR': 'GRC', 'GRAND DUCHY OF LUXEMBOURG': 'LUX', 'GREECE': 'GRC', 'GRENADA': 'GRD', 'GT': 'GTM', 'GUATEMALA': 'GTM', 'GUINEA': 'GIN', 'GUINEA-BISSAU': 'GNB', 'GUYANA': 'GUY', 'GW': 'GNB', 'GY': 'GUY', 'HAITI': 'HTI', 'HASHEMITE KINGDOM OF JORDAN': 'JOR', 'HELLENIC REPUBLIC': 'GRC', 'HN': 'HND', 'HONDURAS': 'HND', 'HR': 'HRV', 'HT': 'HTI', 'HU': 'HUN', 'HUNGARY': 'HUN', 'ICELAND': 'ISL', 'ID': 'IDN', 'IE': 'IRL', 'IL': 'ISR', 'IN': 'IND', 'INDEPENDENT STATE OF PAPUA NEW GUINEA': 'PNG', 'INDEPENDENT STATE OF SAMOA': 'WSM', 'INDIA': 'IND', 'INDONESIA': 'IDN', 'IQ': 'IRQ', 'IR': 'IRN', 'IRAN': 'IRN', 'IRAN (ISLAMIC REPUBLIC OF)': 'IRN', 'IRAN, ISLAMIC REPUBLIC OF': 'IRN', 'IRAQ': 'IRQ', 'IRELAND': 'IRL', 'IS': 'ISL', 'ISLAMIC REPUBLIC OF AFGHANISTAN': 'AFG', 'ISLAMIC REPUBLIC OF IRAN': 'IRN', 'ISLAMIC REPUBLIC OF MAURITANIA': 'MRT', 'ISLAMIC REPUBLIC OF PAKISTAN': 'PAK', 'ISRAEL': 'ISR', 'IT': 'ITA', 'ITALIAN REPUBLIC': 'ITA', 'ITALY': 'ITA', 'JAMAICA': 'JAM', 'JAPAN': 'JPN', 'JM': 'JAM', 'JO': 'JOR', 'JORDAN': 'JOR', 'JP': 'JPN', 'KAZAKHSTAN': 'KAZ', 'KE': 'KEN', 'KENYA': 'KEN', 'KG': 'KGZ', 'KH': 'KHM', 'KI': 'KIR', 'KINGDOM OF BAHRAIN': 'BHR', 'KINGDOM OF BELGIUM': 'BEL', 'KINGDOM OF BHUTAN': 'BTN', 'KINGDOM OF CAMBODIA': 'KHM', 'KINGDOM OF DENMARK': 'DNK', 'KINGDOM OF ESWATINI': 'SWZ', 'KINGDOM OF LESOTHO': 'LSO', 'KINGDOM OF MOROCCO': 'MAR', 'KINGDOM OF NORWAY': 'NOR', 'KINGDOM OF SAUDI ARABIA': 'SAU', 'KINGDOM OF SPAIN': 'ESP', 'KINGDOM OF SWEDEN': 'SWE', 'KINGDOM OF THAILAND': 'THA', 'KINGDOM OF THE NETHERLANDS': 'NLD', 'KINGDOM OF TONGA': 'TON', 'KIRIBATI': 'KIR', 'KM': 'COM', 'KN': 'KNA', "KOREA, DEMOCRATIC PEOPLE'S REPUBLIC OF": 'PRK', 'KOREA, REPUBLIC OF': 'KOR', 'KP': 'PRK', 'KR': 'KOR', 'KUWAIT': 'KWT', 'KW': 'KWT', 'KYRGYZ REPUBLIC': 'KGZ', 'KYRGYZSTAN': 'KGZ', 'KZ': 'KAZ', 'LA': 'LAO', 'LAO PDR': 'LAO', "LAO PEOPLE'S DEMOCRATIC REPUBLIC": 'LAO', 'LAOS': 'LAO', 'LATVIA': 'LVA', 'LB': 'LBN', 'LC': 'LCA', 'LEBANESE REPUBLIC': 'LBN', 'LEBANON': 'LBN', 'LESOTHO': 'LSO', 'LIBERIA': 'LBR', 'LIBYA': 'LBY', 'LITHUANIA': 'LTU', 'LK': 'LKA', 'LR': 'LBR', 'LS': 'LSO', 'LT': 'LTU', 'LU': 'LUX', 'LUXEMBOURG': 'LUX', 'LV': 'LVA', 'LY': 'LBY', 'MA': 'MAR', 'MADAGASCAR': 'MDG', 'MALAWI': 'MWI', 'MALAYSIA': 'MYS', 'MALDIVES': 'MDV', 'MALI': 'MLI', 'MALTA': 'MLT', 'MARSHALL ISLANDS': 'MHL', 'MAURITANIA': 'MRT', 'MAURITIUS': 'MUS', 'MC': 'MCO', 'MD': 'MDA', 'ME': 'MNE', 'MEXICO': 'MEX', 'MG': 'MDG', 'MH': 'MHL', 'MICRONESIA (FEDERATED STATES OF)': 'FSM', 'MICRONESIA, FEDERATED STATES OF': 'FSM', 'MK': 'MKD', 'ML': 'MLI', 'MM': 'MMR', 'MN': 'MNG', 'MOLDOVA': 'MDA', 'MOLDOVA, REPUBLIC OF': 'MDA', 'MONACO': 'MCO', 'MONGOLIA': 'MNG', 'MONTENEGRO': 'MNE', 'MOROCCO': 'MAR', 'MOZAMBIQUE': 'MOZ', 'MR': 'MRT', 'MT': 'MLT', 'MU': 'MUS', 'MV': 'MDV', 'MW': 'MWI', 'MX': 'MEX', 'MY': 'MYS', 'MYANMAR': 'MMR', 'MZ': 'MOZ', 'NA': 'NAM', 'NAMIBIA': 'NAM', 'NAURU': 'NRU', 'NE': 'NER', 'NEPAL': 'NPL', 'NETHERLANDS': 'NLD', 'NEW ZEALAND': 'NZL', 'NG': 'NGA', 'NI': 'NIC', 'NICARAGUA': 'NIC', 'NIGER': 'NER', 'NIGERIA': 'NGA', 'NIUE': 'NIU', 'NL': 'NLD', 'NO': 'NOR', 'NORTH KOREA': 'PRK', 'NORTH MACEDONIA': 'MKD', 'NORWAY': 'NOR', 'NP': 'NPL', 'NR': 'NRU', 'NU': 'NIU', 'NZ': 'NZL', 'OCCUPIED PALESTINIAN TERRITORY': 'PSE', 'OM': 'OMN', 'OMAN': 'OMN', 'PA': 'PAN', 'PAKISTAN': 'PAK', 'PALAU': 'PLW', 'PALESTINE': 'PSE', 'PALESTINE, STATE OF': 'PSE', 'PANAMA': 'PAN', 'PAPUA NEW GUINEA': 'PNG', 'PARAGUAY': 'PRY', 'PE': 'PER', "PEOPLE'S DEMOCRATIC REPUBLIC OF ALGERIA": 'DZA', "PEOPLE'S REPUBLIC OF BANGLADESH": 'BGD', "PEOPLE'S REPUBLIC OF CHINA": 'CHN', 'PERU': 'PER', 'PG': 'PNG', 'PH': 'PHL', 'PHILIPPINES': 'PHL', 'PK': 'PAK', 'PL': 'POL', 'PLURINATIONAL STATE OF BOLIVIA': 'BOL', 'POLAND': 'POL', 'PORTUGAL': 'PRT', 'PORTUGUESE REPUBLIC': 'PRT', 'PR': 'PRI', 'PRINCIPALITY OF ANDORRA': 'AND', 'PRINCIPALITY OF MONACO': 'MCO', 'PS': 'PSE', 'PT': 'PRT', 'PUERTO RICO': 'PRI', 'PW': 'PLW', 'PY': 'PRY', 'QA': 'QAT', 'QATAR': 'QAT', 'REPUBLIC OF ALBANIA': 'ALB', 'REPUBLIC OF ANGOLA': 'AGO', 'REPUBLIC OF ARMENIA': 'ARM', 'REPUBLIC OF AUSTRIA': 'AUT', 'REPUBLIC OF AZERBAIJAN': 'AZE', 'REPUBLIC OF BELARUS': 'BLR', 'REPUBLIC OF BENIN': 'BEN', 'REPUBLIC OF BOSNIA AND HERZEGOVINA': 'BIH', 'REPUBLIC OF BOTSWANA': 'BWA', 'REPUBLIC OF BULGARIA': 'BGR', 'REPUBLIC OF BURUNDI': 'BDI', 'REPUBLIC OF CABO VERDE': 'CPV', 'REPUBLIC OF CAMEROON': 'CMR', 'REPUBLIC OF CHAD': 'TCD', 'REPUBLIC OF CHILE': 'CHL', 'REPUBLIC OF COLOMBIA': 'COL', 'REPUBLIC OF COSTA RICA': 'CRI', 'REPUBLIC OF CROATIA': 'HRV', 'REPUBLIC OF CUBA': 'CUB', 'REPUBLIC OF CYPRUS': 'CYP', "REPUBLIC OF CÔTE D'IVOIRE": 'CIV', 'REPUBLIC OF DJIBOUTI': 'DJI', 'REPUBLIC OF ECUADOR': 'ECU', 'REPUBLIC OF EL SALVADOR': 'SLV', 'REPUBLIC OF EQUATORIAL GUINEA': 'GNQ', 'REPUBLIC OF ESTONIA': 'EST', 'REPUBLIC OF FIJI': 'FJI', 'REPUBLIC OF FINLAND': 'FIN', 'REPUBLIC OF GHANA': 'GHA', 'REPUBLIC OF GUATEMALA': 'GTM', 'REPUBLIC OF GUINEA': 'GIN', 'REPUBLIC OF GUINEA-BISSAU': 'GNB', 'REPUBLIC OF GUYANA': 'GUY', 'REPUBLIC OF HAITI': 'HTI', 'REPUBLIC OF HONDURAS': 'HND', 'REPUBLIC OF ICELAND': 'ISL', 'REPUBLIC OF INDIA': 'IND', 'REPUBLIC OF INDONESIA': 'IDN', 'REPUBLIC OF IRAQ': 'IRQ', 'REPUBLIC OF KAZAKHSTAN': 'KAZ', 'REPUBLIC OF KENYA': 'KEN', 'REPUBLIC OF KIRIBATI': 'KIR', 'REPUBLIC OF KOREA': 'KOR', 'REPUBLIC OF LATVIA': 'LVA', 'REPUBLIC OF LIBERIA': 'LBR', 'REPUBLIC OF LITHUANIA': 'LTU', 'REPUBLIC OF MADAGASCAR': 'MDG', 'REPUBLIC OF MALAWI': 'MWI', 'REPUBLIC OF MALDIVES': 'MDV', 'REPUBLIC OF MALI': 'MLI', 'REPUBLIC OF MALTA': 'MLT', 'REPUBLIC OF MAURITIUS': 'MUS', 'REPUBLIC OF MOLDOVA': 'MDA', 'REPUBLIC OF MOZAMBIQUE': 'MOZ', 'REPUBLIC OF MYANMAR': 'MMR', 'REPUBLIC OF NAMIBIA': 'NAM', 'REPUBLIC OF NAURU': 'NRU', 'REPUBLIC OF NICARAGUA': 'NIC', 'REPUBLIC OF NORTH MACEDONIA': 'MKD', 'REPUBLIC OF PALAU': 'PLW', 'REPUBLIC OF PANAMA': 'PAN', 'REPUBLIC OF PARAGUAY': 'PRY', 'REPUBLIC OF PERU': 'PER', 'REPUBLIC OF POLAND': 'POL', 'REPUBLIC OF SAN MARINO': 'SMR', 'REPUBLIC OF SENEGAL': 'SEN', 'REPUBLIC OF SERBIA': 'SRB', 'REPUBLIC OF SEYCHELLES': 'SYC', 'REPUBLIC OF SIERRA LEONE': 'SLE', 'REPUBLIC OF SINGAPORE': 'SGP', 'REPUBLIC OF SLOVENIA': 'SVN', 'REPUBLIC OF SOUTH AFRICA': 'ZAF', 'REPUBLIC OF SOUTH SUDAN': 'SSD', 'REPUBLIC OF SURINAME': 'SUR', 'REPUBLIC OF TAJIKISTAN': 'TJK', 'REPUBLIC OF THE CONGO': 'COG', 'REPUBLIC OF THE GAMBIA': 'GMB', 'REPUBLIC OF THE MARSHALL ISLANDS': 'MHL', 'REPUBLIC OF THE NIGER': 'NER', 'REPUBLIC OF THE PHILIPPINES': 'PHL', 'REPUBLIC OF THE SUDAN': 'SDN', 'REPUBLIC OF TRINIDAD AND TOBAGO': 'TTO', 'REPUBLIC OF TUNISIA': 'TUN', 'REPUBLIC OF TÜRKIYE': 'TUR', 'REPUBLIC OF UGANDA': 'UGA', 'REPUBLIC OF UZBEKISTAN': 'UZB', 'REPUBLIC OF VANUATU': 'VUT', 'REPUBLIC OF YEMEN': 'YEM', 'REPUBLIC OF ZAMBIA': 'ZMB', 'REPUBLIC OF ZIMBABWE': 'ZWE', 'RO': 'ROU', 'ROMANIA': 'ROU', 'RS': 'SRB', 'RU': 'RUS', 'RUSSIA': 'RUS', 'RUSSIAN FEDERATION': 'RUS', 'RW': 'RWA', 'RWANDA': 'RWA', 'RWANDESE REPUBLIC': 'RWA', 'SA': 'SAU', 'SAINT KITTS AND NEVIS': 'KNA', 'SAINT LUCIA': 'LCA', 'SAINT VINCENT AND THE GRENADINES': 'VCT', 'SAMOA': 'WSM', 'SAN MARINO': 'SMR', 'SAO TOME AND PRINCIPE': 'STP', 'SAUDI ARABIA': 'SAU', 'SB': 'SLB', 'SC': 'SYC', 'SD': 'SDN', 'SE': 'SWE', 'SENEGAL': 'SEN', 'SERBIA': 'SRB', 'SEYCHELLES': 'SYC', 'SG': 'SGP', 'SI': 'SVN', 'SIERRA LEONE': 'SLE', 'SINGAPORE': 'SGP', 'SK': 'SVK', 'SL': 'SLE', 'SLOVAK REPUBLIC': 'SVK', 'SLOVAKIA': 'SVK', 'SLOVENIA': 'SVN', 'SM': 'SMR', 'SN': 'SEN', 'SO': 'SOM', 'SOCIALIST REPUBLIC OF VIET NAM': 'VNM', 'SOLOMON ISLANDS': 'SLB', 'SOMALIA': 'SOM', 'SOUTH AFRICA': 'ZAF', 'SOUTH KOREA': 'KOR', 'SOUTH SUDAN': 'SSD', 'SPAIN': 'ESP', 'SR': 'SUR', 'SRI LANKA': 'LKA', 'SS': 'SSD', 'ST': 'STP', 'STATE OF ISRAEL': 'ISR', 'STATE OF KUWAIT': 'KWT', 'STATE OF PALESTINE': 'PSE', 'STATE OF QATAR': 'QAT', 'SUDAN': 'SDN', 'SULTANATE OF OMAN': 'OMN', 'SURINAME': 'SUR', 'SV': 'SLV', 'SWAZILAND': 'SWZ', 'SWEDEN': 'SWE', 'SWISS CONFEDERATION': 'CHE', 'SWITZERLAND': 'CHE', 'SY': 'SYR', 'SYRIA': 'SYR', 'SYRIAN ARAB REPUBLIC': 'SYR', 'SZ': 'SWZ', 'TAJIKISTAN': 'TJK', 'TANZANIA': 'TZA', 'TANZANIA, UNITED REPUBLIC OF': 'TZA', 'TD': 'TCD', 'TG': 'TGO', 'TH': 'THA', 'THAILAND': 'THA', 'THE GAMBIA': 'GMB', 'THE STATE OF ERITREA': 'ERI', 'THE STATE OF PALESTINE': 'PSE', 'TIMOR-LESTE': 'TLS', 'TJ': 'TJK', 'TK': 'TKL', 'TL': 'TLS', 'TM': 'TKM', 'TN': 'TUN', 'TO': 'TON', 'TOGO': 'TGO', 'TOGOLESE REPUBLIC': 'TGO', 'TOKELAU': 'TKL', 'TONGA': 'TON', 'TR': 'TUR', 'TRINIDAD AND TOBAGO': 'TTO', 'TT': 'TTO', 'TUNISIA': 'TUN', 'TURKEY': 'TUR', 'TURKIYE': 'TUR', 'TURKMENISTAN': 'TKM', 'TUVALU': 'TUV', 'TV': 'TUV', 'TZ': 'TZA', 'TÜRKIYE': 'TUR', 'UA': 'UKR', 'UG': 'UGA', 'UGANDA': 'UGA', 'UK': 'GBR', 'UKRAINE': 'UKR', 'UNION OF THE COMOROS': 'COM', 'UNITED ARAB EMIRATES': 'ARE', 'UNITED KINGDOM': 'GBR', 'UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND': 'GBR', 'UNITED MEXICAN STATES': 'MEX', 'UNITED REPUBLIC OF TANZANIA': 'TZA', 'UNITED STATES': 'USA', 'UNITED STATES OF AMERICA': 'USA', 'URUGUAY': 'URY', 'US': 'USA', 'UY': 'URY', 'UZ': 'UZB', 'UZBEKISTAN': 'UZB', 'VANUATU': 'VUT', 'VC': 'VCT', 'VE': 'VEN', 'VENEZUELA': 'VEN', 'VENEZUELA (BOLIVARIAN REPUBLIC OF)': 'VEN', 'VENEZUELA, BOLIVARIAN REPUBLIC OF': 'VEN', 'VIET NAM': 'VNM', 'VIETNAM': 'VNM', 'VN': 'VNM', 'VU': 'VUT', 'WS': 'WSM', 'YE': 'YEM', 'YEMEN': 'YEM', 'ZA': 'ZAF', 'ZAMBIA': 'ZMB', 'ZIMBABWE': 'ZWE', 'ZM': 'ZMB', 'ZW': 'ZWE'}

def pick_column(frame, candidates, required=True):
    def canon(s):
        return re.sub(r"[^a-z0-9]", "", str(s).lower())
    lookup = {}
    for col in frame.columns:
        c=canon(col)
        if c in lookup:
            raise ValueError("Ambiguous columns after header normalization: " + str(col))
        lookup[c]=col
    for name in candidates:
        if canon(name) in lookup:
            return lookup[canon(name)]
    if required:
        raise ValueError(f"Missing expected column {candidates}. Uploaded columns: {list(frame.columns)}. See upload instructions.")
    return None

def normalize_region(value):
    s = str(value).strip().upper()
    lookup={"AFR":"AFR","AFRO":"AFR","AFRICA":"AFR",
            "AMR":"AMR","AMRO":"AMR","AMERICAS":"AMR","REGION OF THE AMERICAS":"AMR",
            "EMR":"EMR","EMRO":"EMR","EASTERN MEDITERRANEAN":"EMR",
            "EUR":"EUR","EURO":"EUR","EUROPE":"EUR",
            "SEAR":"SEAR","SEA":"SEAR","SEARO":"SEAR","SOUTH-EAST ASIA":"SEAR","SOUTHEAST ASIA":"SEAR",
            "WPR":"WPR","WPRO":"WPR","WESTERN PACIFIC":"WPR"}
    return lookup.get(s)

def normalize_country(value):
    s = str(value).strip().upper()
    if s.endswith(".0") and s[:-2].isdecimal():s=s[:-2]
    if s.isdecimal():s=str(int(s))  # numeric UN M49 key
    if s in COUNTRY_TO_REGION:return s
    return COUNTRY_ALIASES.get(s)

def load_uploaded_national():
    path=WORK/"data/raw/who_ecoli_uploaded.csv"
    if not path.is_file():
        raise FileNotFoundError("Uploaded WHO CSV missing in run folder; rerun the upload cell.")
    payload=path.read_bytes()
    if not payload or hashlib.sha256(payload).hexdigest()!=upload_meta["sha256"]:
        raise ValueError("Uploaded CSV is empty or differs from the upload metadata SHA-256.")
    # Keep every header intact; duplicate headers invalidate the file.
    header=next(__import__('csv').reader(io.StringIO(payload.decode('utf-8-sig'))),[])
    if len(header)!=len(set(header)):
        raise ValueError("Duplicate header names in uploaded CSV.")
    frame=pd.read_csv(io.BytesIO(payload),encoding="utf-8-sig",low_memory=False,dtype=str,keep_default_na=False)
    if frame.empty:raise ValueError("Uploaded CSV contains zero rows.")
    original_columns=list(frame.columns)
    original_count=len(frame)
    geo_type=pick_column(frame,["SpatialDimType","DIM_GEO_CODE_TYPE","DIM_GEO_TYPE"],False)
    if geo_type:
        v=frame[geo_type].astype(str).str.upper().str.strip()
        frame=frame[v.isin(["COUNTRY","TERRITORY","AREA","COUNTRIES"]) | v.eq("")]
    time_type=pick_column(frame,["TimeDimType","DIM_TIME_TYPE"],False)
    if time_type:
        v=frame[time_type].astype(str).str.upper().str.strip()
        frame=frame[v.isin(["YEAR","ANNUAL"]) | v.eq("")]
    pub=pick_column(frame,["DIM_PUBLISH_STATE_CODE"],False)
    if pub:
        frame=frame[frame[pub].astype(str).str.upper().str.strip().isin(["PUBLISHED", "", "FINAL"])]
    indicator=pick_column(frame,["IndicatorCode","GHO","INDICATOR_CODE","DIM_INDICATOR_CODE"],False)
    if indicator:
        values=frame[indicator].astype(str).str.upper().str.strip()
        if values.ne("").any():
            accepted=values.isin(["AMR_INFECT_ECOLI","745F475","ECOLI_3GC_BSI"])
            if accepted.any():frame=frame[accepted]
            else:raise ValueError("CSV indicator column contains no recognized WHO E. coli 3GC resistance code. Do not substitute an unrelated indicator.")
    for dim,options in {"DIM_SEX_CODE":("BTSX","TOTAL","ALL","_T"),
                        "DIM_AGE_CODE":("Y_TOTAL","TOTAL","ALL","_T"),
                        "DIM_AGEGROUP_CODE":("Y_TOTAL","TOTAL","ALL","_T")}.items():
        col=pick_column(frame,[dim],False)
        if col:
            v=frame[col].astype(str).str.upper().str.strip()
            if v.isin(options).any():frame=frame[v.isin(options)]
            elif v.ne("").any():raise ValueError(f"Unresolved stratification {dim}; cannot infer overall national observation.")
    country_col=pick_column(frame,["DIM_GEO_CODE_ISO3","GEO_CODE_ISO3","CODE_ISO_3","ISO3",
        "country_code","SpatialDim","DIM_GEO_CODE","DIM_GEO_CODE_M49","DIM_GEO_NAME","country","Country","GeoAreaName"])
    year_col=pick_column(frame,["DIM_TIME_YEAR","DIM_TIME","TimeDim","TIME_PERIOD","year","YEAR"])
    value_col=pick_column(frame,["AMOUNT_N","NumericValue","VAL_NUMERIC","VALUE_NUMERIC","OBS_VALUE",
        "resistance_pct","resistance_percentage","VALUE","Value"])
    region_col=pick_column(frame,["who_region","WHO_REGION","GRP_WHO_REGION","WHO_REGION_CODE","region"],False)
    selected=frame[[country_col,year_col,value_col]+([region_col] if region_col else [])].copy()
    selected.columns=["country_raw","year_raw","pct_raw"]+(["who_region_raw"] if region_col else [])
    selected["country_code"]=selected.country_raw.map(normalize_country)
    selected["year"]=pd.to_numeric(selected.year_raw.replace("",pd.NA),errors="coerce")
    selected["resistance_pct"]=pd.to_numeric(selected.pct_raw.replace({"":"", "No data":"", "NA":"", "N/A":""}),errors="coerce")
    # Explicit input is a percent, NOT a fraction. E.g. 0.4 means 0.4%, never 40%.
    invalid_pct=selected.pct_raw.astype(str).str.strip().ne("") & selected.resistance_pct.isna() & ~selected.pct_raw.str.lower().isin(["na","n/a","no data","null","nan","none","not available"])
    if invalid_pct.any():
        raise ValueError(f"Non-numeric resistance values in uploaded file, examples: {selected.loc[invalid_pct,'pct_raw'].head(5).tolist()}")
    if selected.year.isna().any() or selected.year.mod(1).ne(0).any():
        raise ValueError("Invalid or missing year values; expected integer observation years.")
    if not selected.resistance_pct.dropna().between(0,100).all():
        raise ValueError("Resistance percentages must lie in [0,100], expressed as percentages, not fractions.")
    missing_pct_count=int(selected.resistance_pct.isna().sum())
    unknown=sorted(set(selected.loc[selected.country_code.isna(),"country_raw"].astype(str)))
    if unknown:
        print(f"Warning: {len(unknown)} country identifiers not recognized by bundled lookup; excluded. Examples: {unknown[:15]}")
        if len(unknown)>20:
            print("If many country codes are unknown, inspect the CSV country column mapping.")
    selected=selected.dropna(subset=["country_code","resistance_pct"]).copy()
    selected["year"]=selected.year.astype(int)
    if region_col:
        selected["provided_region"]=selected.who_region_raw.map(normalize_region)
        unknown_regions=selected.who_region_raw.astype(str).str.strip().ne("") & selected.provided_region.isna()
        if unknown_regions.any():
            raise ValueError("Uploaded who_region contains unknown values: " + str(selected.loc[unknown_regions,"who_region_raw"].unique()[:10].tolist()))
        # Region metadata is not authoritative over a known WHO Member State code;
        # conflicting region assignments require explicit inspection.
        inferred=selected.country_code.map(COUNTRY_TO_REGION)
        mismatches=selected.provided_region.notna() & inferred.notna() & selected.provided_region.ne(inferred)
        if mismatches.any():
            raise ValueError("Uploaded regions contradict embedded WHO May-2025 region mapping. Example: " +
               str(selected.loc[mismatches,["country_code","who_region_raw"]].head(8).to_dict("records"))+
               ". Historical region reassignments need explicit handling, not silent overriding.")
        selected["who_region"]=selected.provided_region.fillna(inferred)
    else:
        selected["who_region"]=selected.country_code.map(COUNTRY_TO_REGION)
    selected=selected.dropna(subset=["who_region"])
    if selected.empty:raise ValueError("No valid country/year resistance observations after validation.")
    # Distinct source-strata might share a numeric value; reject ambiguous observations
    # unless the WHO overall filters already resolved stratification.
    records=selected[["country_code","year","resistance_pct","who_region"]].drop_duplicates()
    if records.duplicated(["country_code","year"]).any():
        offenders=records.loc[records.duplicated(["country_code","year"],keep=False),["country_code","year"]].drop_duplicates().head(8)
        raise ValueError("Conflicting or repeated country/year observations; inspect unfiltered strata. Examples: "+str(offenders.to_dict("records")))
    records=records.sort_values(["country_code","year"]).reset_index(drop=True)
    diagnostics={"uploaded_rows":int(original_count),"country_year_rows":len(records),
        "missing_percentage_rows":missing_pct_count,
        "unmapped_country_codes":unknown,"original_columns":original_columns,
        "selected_fields":{"country":country_col,"year":year_col,"percentage":value_col,"region":region_col},
        "percentage_unit":"percent (0–100)"}
    (WORK/"data/processed/input_validation.json").write_text(json.dumps(diagnostics,indent=2),encoding="utf-8")
    print(f"Loaded {len(records)} unique national observations; {records.country_code.nunique()} countries, years {records.year.min()}–{records.year.max()}.")
    print("Input fields:", diagnostics["selected_fields"])
    return records, original_columns

def aggregate_regions(national):
    grouped=national.groupby(["who_region","year"],as_index=False).agg(
        resistance_pct=("resistance_pct","median"),country_count=("country_code","nunique"))
    min_countries=int(settings.get("minimum_reporting_countries",5))
    if min_countries<5:raise ValueError("Minimum countries cannot be lowered below five for this study.")
    coverage=grouped.copy()
    coverage["eligible_country_count"]=coverage.country_count.ge(min_countries)
    coverage["status"]=np.where(coverage.eligible_country_count,"Derived median available",
        f"Suppressed: fewer than {min_countries} reporting countries")
    coverage.loc[~coverage.eligible_country_count,"resistance_pct"]=np.nan
    regional=grouped[grouped.country_count.ge(min_countries)].copy()
    if regional.empty:raise ValueError(
        "No region/year has at least five reporting countries. EDA of the uploaded national file is possible, "
        "but regional forecasting cannot run. Supply full national coverage or a suitable CSV.")
    return regional.sort_values(["who_region","year"]),coverage
'''


In [ ]:
# FIX: Support the actual WHO E. coli CSV format.
# WHO uses RATE_PER_100_N for resistance percentages.

old = '"resistance_pct","resistance_percentage","VALUE","Value"])'

new = '"RATE_PER_100_N","resistance_pct","resistance_percentage","VALUE","Value"])'

if old not in acquisition_source:
    if new in acquisition_source:
        print("WHO CSV column fix is already applied.")
    else:
        raise RuntimeError(
            "Could not find the expected column mapping. "
            "Check the acquisition_source cell."
        )
else:
    acquisition_source = acquisition_source.replace(old, new, 1)
    print("WHO CSV column mapping updated successfully.")

# Verify the updated worker code.
import ast
ast.parse(acquisition_source)

print("Validation passed. RATE_PER_100_N is now supported.")

In [ ]:
validator_assets = {'src/__init__.py': '"""AMR Sentinel: local surveillance analytics. No predictive model fitting."""\n', 'src/clean.py': '"""Strict, explicit adapters for regional WHO chart exports."""\nimport argparse\nimport json\nfrom datetime import datetime\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nimport numpy as np\nimport pandas as pd\n\nREGIONS = {\'AFR\': \'Africa\', \'AMR\': \'Americas\', \'EMR\': \'Eastern Mediterranean\',\n           \'EUR\': \'Europe\', \'SEAR\': \'South-East Asia\', \'WPR\': \'Western Pacific\'}\nINDICATORS = {\'ECOLI_3GC_BSI\': \'Escherichia coli\', \'MRSA_BSI\': \'Staphylococcus aureus\'}\nKEY = [\'year\', \'who_region\', \'indicator_code\', \'statistic_type\']\nSTATISTICS = (\'who_unadjusted_regional_median\', \'derived_country_median\')\n\n\ndef normalize_and_validate(raw, config, filename):\n    """Map named source fields; scale is declared, never guessed from magnitudes."""\n    if not isinstance(config, dict):\n        raise ValueError(\'Mapping JSON must contain an object, not a list or scalar.\')\n    if config.get(\'percentage_scale\') not in (\'percent\', \'fraction\'):\n        raise ValueError(\'percentage_scale must explicitly be percent or fraction; ambiguous scale rejected.\')\n    required = [\'indicator_code\', \'statistic_type\', \'source_id\', \'source_url\',\n                \'source_revision\', \'source_downloaded_at\', \'definition_note\', \'filters\']\n    for field in required:\n        if not config.get(field):\n            raise ValueError(f\'Missing provenance/definition: {field}\')\n    if config[\'indicator_code\'] not in INDICATORS:\n        raise ValueError(\'Unrecognized indicator_code; only ECOLI_3GC_BSI and MRSA_BSI are supported.\')\n    if config[\'statistic_type\'] not in STATISTICS:\n        raise ValueError(\'Unsupported statistic_type: adjusted estimates must remain separate.\')\n    host = urlparse(config[\'source_url\']).hostname or \'\'\n    if not (host == \'who.int\' or host.endswith(\'.who.int\') or host == \'worldhealthorg.shinyapps.io\'):\n        raise ValueError(\'source_url must be a cited official WHO source page.\')\n    try:\n        timestamp = datetime.fromisoformat(config[\'source_downloaded_at\'].replace(\'Z\', \'+00:00\'))\n        if timestamp.tzinfo is None:\n            raise ValueError()\n    except (ValueError, TypeError):\n        raise ValueError(\'source_downloaded_at requires an ISO-8601 timestamp with timezone.\') from None\n    if raw.empty:\n        raise ValueError(\'Input has no observation rows.\')\n    if raw.columns.duplicated().any():\n        raise ValueError(\'Duplicate source column names are ambiguous.\')\n    mapping = config.get(\'columns\', {})\n    allowed = {\'year\', \'who_region\', \'resistance_pct\', \'country_count\', \'numerator_resistant\', \'denominator_tested\'}\n    if not isinstance(mapping, dict) or set(mapping) - allowed:\n        raise ValueError(\'Unsupported mapped fields. Select a single indicator/statistic in the original WHO export.\')\n    if len(set(mapping.values())) != len(mapping):\n        raise ValueError(\'Each mapped source field must have one meaning.\')\n    for field in (\'year\', \'resistance_pct\'):\n        if field not in mapping:\n            raise ValueError(f\'Explicit column mapping required: {field}\')\n    missing = set(mapping.values()) - set(raw.columns)\n    if missing:\n        raise ValueError(f\'Mapped columns missing: {sorted(missing)}. Available: {list(raw.columns)}\')\n    d = raw[list(mapping.values())].rename(columns={v: k for k, v in mapping.items()}).copy()\n    if \'who_region\' not in d:\n        d[\'who_region\'] = config.get(\'who_region\')\n    aliases = {k.casefold(): k for k in REGIONS}\n    aliases.update({v.casefold(): k for k, v in REGIONS.items()})\n    aliases.update({\'south-east asia region\': \'SEAR\', \'western pacific region\': \'WPR\',\n                    \'african region\': \'AFR\', \'european region\': \'EUR\',\n                    \'region of the americas\': \'AMR\', \'eastern mediterranean region\': \'EMR\'})\n    d[\'who_region\'] = d.who_region.astype(str).str.strip().str.casefold().map(aliases)\n    if d.who_region.isna().any():\n        raise ValueError(\'Unrecognized WHO region; country/continent guessing is prohibited.\')\n    y = pd.to_numeric(d.year, errors=\'raise\')\n    if y.isna().any() or (~np.isfinite(y)).any() or (y % 1 != 0).any() or not y.between(1900, datetime.now().year).all():\n        raise ValueError(\'year must contain integer observation years, no future observations.\')\n    d[\'year\'] = y.astype(int)\n    p = pd.to_numeric(d.resistance_pct, errors=\'raise\')\n    upper = 1 if config[\'percentage_scale\'] == \'fraction\' else 100\n    if not p.dropna().between(0, upper).all():\n        raise ValueError(f\'resistance_pct outside declared scale [0,{upper}].\')\n    d[\'resistance_pct\'] = p.astype(float) * (100 if upper == 1 else 1)\n    for name in (\'country_count\', \'numerator_resistant\', \'denominator_tested\'):\n        vals = pd.to_numeric(d[name], errors=\'raise\') if name in d else pd.Series(float(\'nan\'), index=d.index)\n        if (vals.dropna() < 0).any() or (vals.dropna() % 1 != 0).any() or not np.isfinite(vals.dropna()).all():\n            raise ValueError(f\'{name}: count must be nonnegative integer or null.\')\n        d[name] = vals.astype(\'Int64\')\n    if (d.denominator_tested.dropna() == 0).any():\n        raise ValueError(\'denominator_tested must be greater than zero when provided.\')\n    both = d.numerator_resistant.notna() & d.denominator_tested.notna()\n    if (d.loc[both, \'numerator_resistant\'] > d.loc[both, \'denominator_tested\']).any():\n        raise ValueError(\'resistant count exceeds tested count.\')\n    if config[\'statistic_type\'] == \'derived_country_median\':\n        if d.country_count.isna().any() or (d.country_count < 5).any():\n            raise ValueError(\'Derived medians require at least 5 verified reporting countries per row.\')\n        if not config.get(\'derivation_note\') or not config.get(\'region_mapping_source\'):\n            raise ValueError(\'Derived medians require derivation_note and verified region_mapping_source.\')\n    # WHO chart time-series medians are only eligible with >=5 CTAs when known.\n    low = d.country_count.notna() & (d.country_count < 5) & d.resistance_pct.notna()\n    if low.any():\n        raise ValueError(\'Regional medians with fewer than 5 contributing countries are ineligible.\')\n    d[\'country_count_known\'] = d.country_count.notna()\n    d[\'sample_size_known\'] = both\n    for field in required:\n        if field != \'filters\':\n            d[field] = config[field]\n    d[\'infection_type\'] = \'Bloodstream infection\'\n    d[\'pathogen\'] = INDICATORS[config[\'indicator_code\']]\n    d[\'source_file\'] = Path(filename).name\n    d[\'coverage_note\'] = config.get(\'coverage_note\', \'Unknown counts do not establish WHO count-based eligibility.\')\n    d[\'record_type\'] = \'observed\'\n    d = d.drop_duplicates()\n    if d.duplicated(KEY).any():\n        raise ValueError(\'Conflicting duplicate series/year rows. Resolve source ambiguity before importing.\')\n    return d.sort_values(KEY).reset_index(drop=True)\n\n\ndef main():\n    from src.io_sources import import_files, write_processed\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\', nargs=\'+\', required=True, help=\'Untouched WHO regional CSV/XLSX files\')\n    parser.add_argument(\'--mapping\', required=True, help=\'Explicit adapter JSON, applied to all supplied files\')\n    parser.add_argument(\'--root\', default=\'.\', help=\'Project output directory\')\n    args = parser.parse_args()\n    config = json.loads(Path(args.mapping).read_text(encoding=\'utf-8\'))\n    data, manifest, raw_files = import_files([(Path(p).name, Path(p).read_bytes(), config) for p in args.input])\n    write_processed(Path(args.root), data, manifest, raw_files)\n    print(f\'Imported {len(data)} rows. No predictive models were trained.\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'src/analytics.py': '"""Descriptive statistics only; this module trains no forecasting model."""\nimport numpy as np\nimport pandas as pd\nfrom src.clean import KEY, REGIONS\n\n\ndef single_definition(d):\n    for col in (\'indicator_code\', \'statistic_type\'):\n        if d[col].nunique() > 1:\n            raise ValueError(f\'Select a single {col} before calculating a series.\')\n\n\ndef tail_history(d):\n    d = d.dropna(subset=[\'resistance_pct\']).sort_values(\'year\')\n    if d.empty:\n        return d\n    breaks = d.year.diff().ne(1)\n    return d.loc[breaks.cumsum() == breaks.cumsum().iloc[-1]]\n\n\ndef annual_changes(d):\n    d = d.sort_values(KEY).copy()\n    groups = d.groupby([\'who_region\', \'indicator_code\', \'statistic_type\'])\n    d[\'annual_change_pp\'] = groups.resistance_pct.diff().where(groups.year.diff().eq(1))\n    return d\n\n\ndef summarize(d):\n    single_definition(d)\n    if d.who_region.nunique() > 1:\n        raise ValueError(\'Select one WHO region.\')\n    valid = d.dropna(subset=[\'resistance_pct\']).sort_values(\'year\')\n    tail = tail_history(valid)\n    years = valid.year.astype(int).tolist()\n    missing = sorted(set(range(min(years), max(years) + 1)) - set(years)) if years else []\n    slope = None\n    direction = \'Insufficient data\'\n    if len(tail) >= 3:\n        x = tail.year.to_numpy(dtype=float)\n        y = tail.resistance_pct.to_numpy(dtype=float)\n        x = x - x.mean()\n        slope = float(np.dot(x, y - y.mean()) / np.dot(x, x))\n        direction = \'Increasing\' if slope >= 1 else \'Decreasing\' if slope <= -1 else \'Inconclusive/near-flat\'\n    return {\'n_observations\': len(valid), \'last_year\': max(years) if years else None,\n            \'missing_years\': \', \'.join(map(str, missing)), \'consecutive_years\': len(tail),\n            \'years_used\': \', \'.join(tail.year.astype(str)), \'slope_pp_per_year\': slope,\n            \'direction\': direction, \'forecast_eligible\': len(tail) >= 4,\n            \'eligibility_reason\': \'Eligible for exploratory baselines; Ridge eligibility decided in Colab\'\n            if len(tail) >= 4 else \'Insufficient data to forecast: require 4 consecutive observed years ending at last observation\',\n            \'status\': \'Observed\' if years else \'Data unavailable\',\n            \'unknown_country_counts\': int(valid.country_count.isna().sum()) if \'country_count\' in valid else len(valid)}\n\n\ndef quality_report(d):\n    rows = []\n    for (indicator, statistic), g in d.groupby([\'indicator_code\', \'statistic_type\']):\n        for region in REGIONS:\n            s = g[g.who_region == region]\n            rows.append({\'who_region\': region, \'indicator_code\': indicator, \'statistic_type\': statistic,\n                         **summarize(s)})\n    return pd.DataFrame(rows)\n', 'src/io_sources.py': '"""Raw byte preservation, explicit mapping and content-addressed handoff."""\nimport csv\nimport hashlib\nimport io\nimport json\nimport zipfile\nfrom pathlib import Path\n\nimport pandas as pd\nfrom src.clean import KEY, normalize_and_validate\nfrom src.analytics import quality_report\n\n\ndef sha256(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef read_export(name, payload):\n    suffix = Path(name).suffix.lower()\n    if suffix == \'.csv\':\n        header = next(csv.reader(io.StringIO(payload.decode(\'utf-8-sig\'))), [])\n        if len(header) != len(set(header)):\n            raise ValueError(\'Duplicate source column names are ambiguous.\')\n        return pd.read_csv(io.BytesIO(payload), encoding=\'utf-8-sig\')\n    if suffix == \'.xlsx\':\n        try:\n            header = pd.read_excel(io.BytesIO(payload), header=None, nrows=1).iloc[0].dropna().tolist()\n            if len(header) != len(set(header)):\n                raise ValueError(\'Duplicate source column names are ambiguous.\')\n            return pd.read_excel(io.BytesIO(payload))\n        except (zipfile.BadZipFile, IndexError, OSError) as exc:\n            raise ValueError(f\'Cannot read XLSX export: {exc}\') from exc\n    raise ValueError(\'Only UTF-8 comma-separated CSV or XLSX is supported. Export or convert explicitly.\')\n\n\ndef import_files(items):\n    tables, files, manifest = [], {}, {\'manifest_version\': 1, \'files\': []}\n    for name, payload, config in items:\n        name = Path(name).name\n        if name in files and files[name] != payload:\n            raise ValueError(\'Conflicting input filenames; rename files before importing.\')\n        raw = read_export(name, payload)\n        d = normalize_and_validate(raw, config, name)\n        d[\'source_sha256\'] = sha256(payload)\n        tables.append(d)\n        files[name] = payload\n        manifest[\'files\'].append({\'filename\': name, \'sha256\': sha256(payload),\n                                  \'original_columns\': list(raw.columns), \'adapter\': config})\n    if not tables:\n        raise ValueError(\'Select at least one authentic WHO file.\')\n    d = pd.concat(tables, ignore_index=True).drop_duplicates()\n    if d.duplicated(KEY).any():\n        raise ValueError(\'Conflicting duplicate rows across files; do not mix revisions or source definitions.\')\n    # A target cannot change source or methodology between years.\n    for _, g in d.groupby([\'who_region\', \'indicator_code\', \'statistic_type\']):\n        for col in (\'source_id\', \'source_revision\', \'definition_note\', \'source_url\'):\n            if g[col].nunique() > 1:\n                raise ValueError(f\'Source/definition change in {col}; import as a separate study.\')\n    return d.sort_values(KEY).reset_index(drop=True), manifest, files\n\n\ndef observations_csv(d):\n    return d.sort_values(KEY).to_csv(index=False, lineterminator=\'\\n\').encode(\'utf-8\')\n\n\ndef data_dictionary():\n    return \'\'\'# Normalized data dictionary\n\nRows are observed annual WHO regional statistics, never projections.\nyear: integer observation year. who_region: AFR/AMR/EMR/EUR/SEAR/WPR.\nindicator_code: ECOLI_3GC_BSI or MRSA_BSI. infection_type: Bloodstream infection.\npathogen: canonical organism. resistance_pct: percentage in [0,100], null stays null.\nstatistic_type: who_unadjusted_regional_median or separately labeled derived_country_median.\nsource_id, source_url, source_file, source_downloaded_at, source_revision,\nsource_sha256: trace input and release. definition_note: exact metric meaning.\ncountry_count: nullable reporting-country count; country_count_known: bool.\nnumerator_resistant, denominator_tested: nullable compatible infection counts.\nsample_size_known: both compatible count fields supplied; no count eligibility claim.\ncoverage_note: source limitations. record_type: observed.\n\nTransformations: explicitly mapped fields, documented WHO region aliases,\ndeclared fraction-to-percent scaling only, exact redundant duplicates dropped,\nconflicting duplicates rejected. Nulls never become zero. No interpolation.\nDerived medians need >=5 countries and documented mapping/selection, and never\nreproduce WHO\'s count-based eligibility without the required infection counts.\nUnknown country/testing counts do not demonstrate WHO eligibility thresholds.\n\nSource: WHO GLASS https://worldhealthorg.shinyapps.io/glass-dashboard/\nPrimary definition: https://data.who.int/indicators/i/918081E/745F475\nSecondary definition: https://data.who.int/indicators/i/918081E/5DD9606\n\'\'\'\n\n\ndef write_processed(root, d, manifest, raw_files):\n    raw_dir, processed = root / \'data/raw\', root / \'data/processed\'\n    raw_dir.mkdir(parents=True, exist_ok=True)\n    processed.mkdir(parents=True, exist_ok=True)\n    # Refuse to overwrite an untouched raw input with different bytes.\n    for name, payload in raw_files.items():\n        target = raw_dir / name\n        if target.exists() and target.read_bytes() != payload:\n            raise ValueError(f\'Raw file already exists with different bytes: {name}\')\n    for name, payload in raw_files.items():\n        (raw_dir / name).write_bytes(payload)\n    (root / \'data/source_manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    # Parquet is an OPTIONAL convenience export. CSV is always authoritative.\n    # Colab often lacks a compatible pyarrow wheel; do not break analysis for it.\n    try:\n        import pyarrow  # noqa: F401\n    except ImportError:\n        pass\n    else:\n        d.to_parquet(processed / \'regional_resistance.parquet\', index=False)\n    (processed / \'regional_resistance.csv\').write_bytes(observations_csv(d))\n    quality_report(d).to_csv(processed / \'data_quality.csv\', index=False)\n    (processed / \'data_dictionary.md\').write_text(data_dictionary(), encoding=\'utf-8\')\n\n\ndef input_bundle(d, manifest):\n    payload = observations_csv(d)\n    contract = {\'schema_version\': 1, \'observations_sha256\': sha256(payload),\n                \'manifest\': manifest, \'note\': \'WHO observations only; run modeling in Google Colab.\'}\n    output = io.BytesIO()\n    with zipfile.ZipFile(output, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        archive.writestr(\'observations.csv\', payload)\n        archive.writestr(\'input_metadata.json\', json.dumps(contract, indent=2))\n    return output.getvalue()\n\n\ndef load_saved(root):\n    """Rebuild normalized rows from hash-verified untouched source files."""\n    manifest = json.loads((root / \'data/source_manifest.json\').read_text(encoding=\'utf-8\'))\n    items = []\n    for entry in manifest[\'files\']:\n        name = entry[\'filename\']\n        if Path(name).name != name:\n            raise ValueError(\'Invalid source filename in manifest.\')\n        payload = (root / \'data/raw\' / name).read_bytes()\n        if sha256(payload) != entry[\'sha256\']:\n            raise ValueError(f\'Raw checksum changed: {name}. Re-import explicitly.\')\n        items.append((name, payload, entry[\'adapter\']))\n    return import_files(items)\n', 'src/results.py': '"""Validate Colab CSV/JSON outputs. Never import sklearn or unpickle models."""\nimport io\nimport json\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom src.io_sources import observations_csv, sha256\nfrom src.analytics import tail_history\nfrom src.clean import REGIONS, STATISTICS\n\nMODELS = (\'naive\', \'linear\', \'ridge\')\nRESULT_FILES = {\'backtests.csv\', \'model_metrics.csv\', \'forecasts.csv\', \'model_metadata.json\'}\n\n\ndef score_predictions(rows):\n    """Three-way comparison on identical cases; baseline-only comparison if no Ridge."""\n    if rows.empty:\n        return pd.DataFrame(columns=[\'model\', \'mae_pp\', \'rmse_pp\', \'n\', \'comparison\'])\n    models = list(MODELS) if rows.ridge_pred.notna().any() else [\'naive\', \'linear\']\n    shared = rows.dropna(subset=[\'actual\'] + [m + \'_pred\' for m in models])\n    scores = []\n    for model in models:\n        if shared.empty:\n            continue\n        err = shared[model + \'_pred\'] - shared.actual\n        scores.append({\'model\': model, \'mae_pp\': float(err.abs().mean()),\n                       \'rmse_pp\': float(np.sqrt((err ** 2).mean())), \'n\': len(shared),\n                       \'comparison\': \'matched_three_models\' if len(models) == 3 else \'matched_baselines\'})\n    return pd.DataFrame(scores, columns=[\'model\', \'mae_pp\', \'rmse_pp\', \'n\', \'comparison\'])\n\n\ndef load_results(payload, observations, expected_hash=None):\n    expected_hash = expected_hash or sha256(observations_csv(observations))\n    try:\n        with zipfile.ZipFile(io.BytesIO(payload)) as archive:\n            entries = archive.infolist()\n            if len({e.filename for e in entries}) != len(entries) or any(e.filename not in RESULT_FILES for e in entries):\n                raise ValueError(\'Unexpected archive entries; use the Colab results ZIP, without models.\')\n            if sum(e.file_size for e in entries) > 25_000_000:\n                raise ValueError(\'Result archive exceeds 25 MB.\')\n            metadata = json.loads(archive.read(\'model_metadata.json\'))\n            if metadata.get(\'observations_sha256\') != expected_hash:\n                raise ValueError(\'Results dataset does not match the currently imported observations.\')\n            if set(e.filename for e in entries) != RESULT_FILES:\n                raise ValueError(\'Missing result files in Colab archive.\')\n            tables = {name: pd.read_csv(io.BytesIO(archive.read(name))) for name in RESULT_FILES if name.endswith(\'.csv\')}\n    except (zipfile.BadZipFile, KeyError, json.JSONDecodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as exc:\n        raise ValueError(f\'Invalid Colab results bundle: {exc}\') from exc\n    if metadata.get(\'schema_version\') != 1 or metadata.get(\'execution_environment\') != \'google_colab\':\n        raise ValueError(\'Results require the supported Colab metadata contract.\')\n    if metadata.get(\'indicator_code\') != \'ECOLI_3GC_BSI\' or metadata.get(\'statistic_type\') not in STATISTICS:\n        raise ValueError(\'Unrecognized modeling target or statistic.\')\n    target = observations[(observations.indicator_code == metadata[\'indicator_code\']) &\n                          (observations.statistic_type == metadata[\'statistic_type\'])]\n    backtests, forecasts = tables[\'backtests.csv\'], tables[\'forecasts.csv\']\n    base = [\'who_region\', \'indicator_code\', \'statistic_type\', \'target_year\', \'record_type\']\n    diagnostics = [model + suffix for model in MODELS for suffix in (\'_raw\', \'_clipped\')]\n    contracts = [(backtests, base + [\'actual\', \'naive_pred\', \'linear_pred\', \'ridge_pred\', \'phase\', \'train_end_year\'] + diagnostics),\n                 (forecasts, base + [\'model\', \'prediction\', \'raw_prediction\', \'clipped\', \'last_observed_year\'])]\n    for table, columns in contracts:\n        if not set(columns) <= set(table.columns):\n            raise ValueError(f\'Result columns missing: {sorted(set(columns) - set(table.columns))}\')\n        if not table.who_region.isin(REGIONS).all():\n            raise ValueError(\'Unknown result region.\')\n        if not table.indicator_code.eq(metadata[\'indicator_code\']).all() or not table.statistic_type.eq(metadata[\'statistic_type\']).all():\n            raise ValueError(\'Mixed target definitions in results.\')\n        for column in [\'target_year\']:\n            vals = pd.to_numeric(table[column], errors=\'raise\')\n            if vals.isna().any() or (vals % 1 != 0).any():\n                raise ValueError(\'Result years must be integers.\')\n        for column in (\'prediction\', \'actual\', \'naive_pred\', \'linear_pred\', \'ridge_pred\'):\n            if column in table:\n                table[column] = pd.to_numeric(table[column], errors=\'raise\')\n                if not table[column].dropna().between(0, 100).all():\n                    raise ValueError(\'Result percentage outside [0,100].\')\n    if backtests.duplicated([\'who_region\', \'target_year\']).any() or forecasts.duplicated([\'who_region\', \'target_year\', \'model\']).any():\n        raise ValueError(\'Duplicate result cases.\')\n    holdout = int(metadata[\'holdout_year\'])\n    if target.dropna(subset=[\'resistance_pct\']).empty or holdout != int(target.dropna(subset=[\'resistance_pct\']).year.max()):\n        raise ValueError(\'Final holdout must equal the latest target observation year.\')\n    expected_cases = set()\n    for obs in target.dropna(subset=[\'resistance_pct\']).itertuples():\n        prior = tail_history(target[(target.who_region == obs.who_region) & (target.year < obs.year)])\n        if len(prior) >= 4 and prior.year.max() == obs.year - 1:\n            expected_cases.add((obs.who_region, int(obs.year)))\n    supplied_cases = set(zip(backtests.who_region, backtests.target_year.astype(int)))\n    if supplied_cases != expected_cases:\n        raise ValueError(\'Backtest cases do not match all eligible chronological observations; missing or ineligible cases found.\')\n    if not backtests.phase.isin([\'validation\', \'holdout\']).all():\n        raise ValueError(\'Unknown evaluation phase.\')\n    for row in backtests.itertuples():\n        series = target[target.who_region == row.who_region]\n        actual = series[series.year == row.target_year].resistance_pct.dropna()\n        if len(actual) != 1 or not np.isclose(actual.iloc[0], row.actual):\n            raise ValueError(\'Backtest actual does not match observed WHO value.\')\n        prior = tail_history(series[series.year < row.target_year])\n        if row.train_end_year != row.target_year - 1 or len(prior) < 4 or prior.year.max() != row.target_year - 1:\n            raise ValueError(\'Backtest chronology or history eligibility is invalid.\')\n        if (row.phase == \'holdout\') != (row.target_year == holdout) or row.target_year > holdout:\n            raise ValueError(\'Invalid holdout partition.\')\n        if row.record_type != \'backtest_prediction\':\n            raise ValueError(\'Backtests must be labeled as predictions.\')\n        for model in MODELS:\n            predicted, raw = getattr(row, model + \'_pred\'), getattr(row, model + \'_raw\')\n            flag = str(getattr(row, model + \'_clipped\')).lower()\n            if flag not in (\'true\', \'false\'):\n                raise ValueError(\'Clipping flags must be boolean.\')\n            if pd.isna(predicted):\n                if model != \'ridge\' or not pd.isna(raw) or flag != \'false\':\n                    raise ValueError(\'Only unavailable Ridge may have a missing backtest prediction.\')\n                continue\n            if not np.isfinite(raw) or not np.isclose(np.clip(raw, 0, 100), predicted) or (flag == \'true\') != (raw < 0 or raw > 100):\n                raise ValueError(\'Backtest bounding diagnostics do not match.\')\n    for row in forecasts.itertuples():\n        series = target[target.who_region == row.who_region].dropna(subset=[\'resistance_pct\'])\n        if series.empty or len(tail_history(series)) < 4:\n            raise ValueError(\'Forecast history is ineligible.\')\n        if row.last_observed_year != series.year.max() or row.target_year != row.last_observed_year + 1:\n            raise ValueError(\'Forecast must be one year after the last observation.\')\n        if row.model not in MODELS or row.record_type != \'experimental_forecast\':\n            raise ValueError(\'Unrecognized forecast model or record type.\')\n        if not np.isfinite(row.raw_prediction) or not np.isclose(np.clip(row.raw_prediction, 0, 100), row.prediction):\n            raise ValueError(\'Forecast bounding diagnostics do not match.\')\n        if str(row.clipped).lower() not in (\'true\', \'false\'):\n            raise ValueError(\'Forecast clipping flags must be boolean.\')\n        is_clipped = str(row.clipped).lower() == \'true\'\n        if is_clipped != (row.raw_prediction < 0 or row.raw_prediction > 100):\n            raise ValueError(\'Forecast clipping flag does not match raw prediction.\')\n    # Recompute scores locally using arithmetic, never trust uploaded metric claims.\n    scores = score_predictions(backtests[backtests.phase == \'holdout\'])\n    provided = tables[\'model_metrics.csv\']\n    if not set(scores.columns) <= set(provided.columns) or len(provided) != len(scores):\n        raise ValueError(\'Metric file schema or number of models does not match held-out cases.\')\n    if len(scores):\n        a, b = scores.sort_values(\'model\'), provided.sort_values(\'model\')\n        if a.model.tolist() != b.model.tolist() or not np.allclose(a[[\'mae_pp\', \'rmse_pp\', \'n\']], b[[\'mae_pp\', \'rmse_pp\', \'n\']]):\n            raise ValueError(\'Uploaded metrics disagree with matched holdout errors.\')\n    return {\'metadata\': metadata, \'backtests\': backtests, \'forecasts\': forecasts, \'metrics\': scores}\n\n\ndef save_results(root, payload, observations):\n    """Persist only verified plain-data outputs; never extract arbitrary ZIP paths."""\n    verified = load_results(payload, observations)\n    output = root / \'results\'\n    output.mkdir(exist_ok=True, parents=True)\n    for name in (\'backtests\', \'forecasts\', \'metrics\'):\n        filename = \'model_metrics.csv\' if name == \'metrics\' else name + \'.csv\'\n        verified[name].to_csv(output / filename, index=False)\n    (output / \'model_metadata.json\').write_text(json.dumps(verified[\'metadata\'], indent=2), encoding=\'utf-8\')\n    return verified\n\n\ndef load_saved_results(root, observations):\n    buffer = io.BytesIO()\n    with zipfile.ZipFile(buffer, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for name in RESULT_FILES:\n            archive.writestr(name, (root / \'results\' / name).read_bytes())\n    return load_results(buffer.getvalue(), observations)\n\n\ndef main():\n    import argparse\n    from pathlib import Path\n    from src.io_sources import load_saved\n    parser = argparse.ArgumentParser(description=\'Validate Colab results and save CSV/JSON; no model fitting.\')\n    parser.add_argument(\'--input\', required=True, help=\'Returned amr_colab_results.zip\')\n    parser.add_argument(\'--root\', default=\'.\')\n    args = parser.parse_args()\n    root = Path(args.root)\n    observations, _, _ = load_saved(root)\n    save_results(root, Path(args.input).read_bytes(), observations)\n    print(\'Verified Colab results saved. No predictive model was loaded or trained.\')\n\n\nif __name__ == \'__main__\':\n    main()\n'}


## 5. Exploratory Data Analysis (EDA)

The next code cell defines a **reproducible EDA stage** (executed in the worker after CSV validation, before ML). It exports missing-data and coverage tables, national summary statistics, region-year observation counts and actual PNG charts: distribution, region boxplots, region trends, participation heatmap, latest-year country extremes and a histogram of year-to-year changes. EDA uses **only observed data**; forecasts are not included. The full tables and charts will be in the result bundle and report.


In [ ]:
eda_source = r'''
# Exploratory data analysis: actual observed national data only.
eda_dir=WORK/"reports/eda"
eda_dir.mkdir(parents=True,exist_ok=True)
summary=national.resistance_pct.describe(percentiles=[.1,.25,.5,.75,.9]).rename("resistance_pct").to_frame()
summary.to_csv(eda_dir/"national_descriptive_stats.csv")
by_region=national.groupby("who_region").agg(national_rows=("resistance_pct","size"),
    reporting_countries=("country_code","nunique"),min_year=("year","min"),max_year=("year","max"),
    mean_pct=("resistance_pct","mean"),median_pct=("resistance_pct","median"),
    std_pct=("resistance_pct","std")).reset_index()
by_region.to_csv(eda_dir/"region_summary.csv",index=False)
by_year=national.groupby("year").agg(national_rows=("resistance_pct","size"),
    reporting_countries=("country_code","nunique"),median_pct=("resistance_pct","median")).reset_index()
by_year.to_csv(eda_dir/"year_summary.csv",index=False)
region_year=national.groupby(["who_region","year"]).agg(n_countries=("country_code","nunique"),
    median_pct=("resistance_pct","median"),min_pct=("resistance_pct","min"),
    max_pct=("resistance_pct","max")).reset_index()
region_year.to_csv(eda_dir/"region_year_summary_all.csv",index=False)
# Important: a missing year is not a zero. Identify gaps in each region explicitly.
gaps=[]
for region,g in coverage.groupby("who_region"):
    years=set(g.year.astype(int))
    for y in range(int(g.year.min()),int(g.year.max())+1):
        if y not in years:gaps.append({"who_region":region,"year":y,"issue":"no_reporting_observation"})
        elif not bool(g.loc[g.year.eq(y),"eligible_country_count"].iloc[0]):
            gaps.append({"who_region":region,"year":y,"issue":"fewer_than_5_countries"})
pd.DataFrame(gaps,columns=["who_region","year","issue"]).to_csv(eda_dir/"coverage_gaps.csv",index=False)

plt.rcParams.update({"figure.figsize":(10,5),"axes.spines.top":False,
                     "axes.spines.right":False,"axes.grid":True,"grid.alpha":.15})
palette={"AFR":"#087f8c","AMR":"#c56a3c","EMR":"#657593",
         "EUR":"#586c50","SEAR":"#8b5880","WPR":"#a38b31"}
eda_figures=[]
def save_eda(fig,filename):
    fig.tight_layout()
    target=eda_dir/filename
    fig.savefig(target,dpi=150,bbox_inches="tight")
    plt.close(fig)
    eda_figures.append(target)

fig,ax=plt.subplots(figsize=(9,4.5))
ax.hist(national.resistance_pct,bins=np.linspace(0,100,21),color="#087f8c",edgecolor="white")
ax.set(title="National resistance distribution (observations)",xlabel="Resistance (%)",ylabel="Country-year records")
save_eda(fig,"eda_01_distribution.png")

fig,ax=plt.subplots(figsize=(10,5))
present=[r for r in REGIONS if (national.who_region==r).any()]
ax.boxplot([national.loc[national.who_region.eq(r),"resistance_pct"].to_numpy() for r in present],tick_labels=present)
ax.set(title="National resistance by WHO region (pooled across years)",xlabel="WHO region",ylabel="Resistance (%)")
save_eda(fig,"eda_02_regional_boxplot.png")

fig,ax=plt.subplots(figsize=(10,5))
for region in REGIONS:
    g=regional[regional.who_region.eq(region)].sort_values("year")
    if not g.empty:
        s=g.set_index("year").resistance_pct.reindex(range(int(g.year.min()),int(g.year.max())+1))
        ax.plot(s.index,s.values,marker="o",label=region,color=palette[region])
ax.set(title="Eligible regional medians: observed historical trends",xlabel="Observation year",ylabel="Resistance (%)")
ax.legend(ncol=3)
save_eda(fig,"eda_03_regional_trends.png")

pivot=coverage.pivot(index="who_region",columns="year",values="country_count").reindex(REGIONS)
fig,ax=plt.subplots(figsize=(11,4.3))
mat=pivot.to_numpy(dtype=float)
im=ax.imshow(np.ma.masked_invalid(mat),aspect="auto",cmap="YlGnBu")
ax.set_yticks(range(len(REGIONS)),REGIONS)
ax.set_xticks(range(len(pivot.columns)),[str(x) for x in pivot.columns],rotation=45)
fig.colorbar(im,ax=ax,label="Reporting countries")
ax.set(title="Surveillance participation by region and year")
save_eda(fig,"eda_04_coverage_heatmap.png")

fig,ax=plt.subplots(figsize=(9,4.5))
ax.plot(by_year.year,by_year.reporting_countries,marker="o",color="#087f8c")
ax.set(title="Number of countries reporting each year",xlabel="Year",ylabel="Unique countries")
save_eda(fig,"eda_05_reporting_countries.png")

last_year=int(national.year.max())
latest=national[national.year.eq(last_year)].sort_values("resistance_pct")
if len(latest):
    extremes=pd.concat([latest.head(8),latest.tail(8)]).drop_duplicates("country_code").sort_values("resistance_pct")
    extremes[["country_code","who_region","year","resistance_pct"]].to_csv(eda_dir/"latest_year_country_extremes.csv",index=False)
    fig,ax=plt.subplots(figsize=(9,6))
    ax.barh(extremes.country_code,extremes.resistance_pct,color="#657593")
    ax.set(xlabel="Resistance (%)",title=f"Selected low/high reporting countries ({last_year}; descriptive only)")
    save_eda(fig,"eda_06_country_extremes.png")

annual=national.sort_values(["country_code","year"]).copy()
annual["previous_year"]=annual.groupby("country_code").year.shift(1)
annual["previous_pct"]=annual.groupby("country_code").resistance_pct.shift(1)
annual["change_pp"]=np.where(annual.year.sub(annual.previous_year).eq(1),
                            annual.resistance_pct.sub(annual.previous_pct),np.nan)
changes=annual.dropna(subset=["change_pp"])[["country_code","who_region","year","change_pp"]]
changes.to_csv(eda_dir/"year_on_year_country_changes.csv",index=False)
if not changes.empty:
    fig,ax=plt.subplots(figsize=(9,4.5))
    ax.hist(changes.change_pp,bins=min(25,max(5,int(np.sqrt(len(changes))))),color="#c56a3c",edgecolor="white")
    ax.set(title="Annual country-level change (consecutive observations only)",xlabel="Change in percentage points",ylabel="Country-year pairs")
    save_eda(fig,"eda_07_annual_change_distribution.png")

eda_summary={"national_observations":int(len(national)),"countries":int(national.country_code.nunique()),
             "regions_observed":int(national.who_region.nunique()),
             "observation_years":[int(national.year.min()),int(national.year.max())],
             "eligible_region_years":int(len(regional)),
             "coverage_gap_records":len(gaps),
             "country_year_pairs_with_change":int(len(changes))}
(eda_dir/"eda_summary.json").write_text(json.dumps(eda_summary,indent=2),encoding="utf-8")
print("EDA complete:",eda_summary,"| figures:",len(eda_figures))
'''


## 6. Modeling, evaluation and reporting definitions

Chronological validation and experimental one-year forecasting use naïve, linear and pooled Ridge models. A historical year is never randomly shuffled into training. Ridge is fitted only if enough prior training rows and regions exist. The code below also assembles EDA, forecasts, source manifests and a self-contained HTML report. Fitted models and model metrics are saved in the ZIP outputs.


In [ ]:
worker_prelude = '\nimport os, sys, io, json, hashlib, zipfile, platform, re, html, traceback\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nif os.environ.get("AMR_COLAB_AUTHORIZED") != "1" or not Path("/content").is_dir():\n    raise RuntimeError("This experiment is authorized only inside Google Colab.")\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport sklearn\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import Ridge\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nWORK = Path(__file__).resolve().parent\nfor name in ("data/raw", "data/processed", "results", "models", "reports", "reports/figures"):\n    (WORK / name).mkdir(parents=True, exist_ok=True)\nsettings = json.loads((WORK / "run_settings.json").read_text(encoding="utf-8"))\nupstream_sources, acquisition_log = [], []\nupload_meta=json.loads((WORK/"upload_metadata.json").read_text(encoding="utf-8"))\n\ndef display(value):\n    print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else str(value))\n'
processing_source = 'from src.io_sources import import_files, write_processed, observations_csv, input_bundle\nfrom src.analytics import quality_report, annual_changes\nnational, original_columns = load_uploaded_national()\nrevision = "User-uploaded national CSV; publication revision not independently verified"\nmapping_url = WHO_MAPPING_SOURCE\nsource_download_url = "local upload: " + upload_meta["original_filename"]\ninput_bytes=(WORK/"data/raw/who_ecoli_uploaded.csv").read_bytes()\nupstream_sources.append({\n    "filename":"who_ecoli_uploaded.csv", "original_filename":upload_meta["original_filename"],\n    "source_url":INDICATOR_PAGE,"source_origin":"user_uploaded_unverified_provenance",\n    "downloaded_at":upload_meta["uploaded_utc"],"sha256":hashlib.sha256(input_bytes).hexdigest(),\n    "source_revision":revision,\n})\nacquisition_log.append({"stage":"local_uploaded_csv","status":"validated",\n    "filename":upload_meta["original_filename"],"sha256":upload_meta["sha256"]})\nnational.to_csv(WORK/"data/processed/national_observations.csv",index=False)\nregional,coverage=aggregate_regions(national)\ncoverage.to_csv(WORK/"data/processed/country_coverage_all_years.csv",index=False)\ncomposition=national.groupby(["who_region","year"],as_index=False).agg(\n    country_codes=("country_code",lambda x:";".join(sorted(x.unique()))))\ncomposition.to_csv(WORK/"data/processed/reporting_country_composition.csv",index=False)\nderived_bytes=regional.to_csv(index=False,lineterminator="\\n").encode("utf-8")\nadapter={\n    "columns":{"year":"year","who_region":"who_region","resistance_pct":"resistance_pct",\n        "country_count":"country_count"},\n    "percentage_scale":"percent","indicator_code":"ECOLI_3GC_BSI",\n    "statistic_type":"derived_country_median","source_id":"S4",\n    "source_url":INDICATOR_PAGE,"source_revision":revision,\n    "source_downloaded_at":upload_meta["uploaded_utc"],\n    "definition_note":"Derived unweighted median of available national E. coli 3GC bloodstream resistance percentages per WHO region/year; >=5 reporting countries.",\n    "filters":{"indicator":"AMR_INFECT_ECOLI / 745F475","geography":"national","time":"annual",\n               "aggregation":"unweighted country median","minimum_country_count":5},\n    "derivation_note":"Single uploaded source CSV, unique country/year percentages; bundled WHO regional mapping; no weighting/interpolation; no testing-count eligibility claim.",\n    "region_mapping_source":mapping_url,\n    "coverage_note":"Regions use bundled WHO May-2025 Member State mapping (including Indonesia in WPR); historical membership may differ. Country coverage varies and infection counts unknown. CSV provenance supplied by user; verify with WHO if publishing."\n}\ndata,manifest,raw_files=import_files([("derived_regional_input.csv",derived_bytes,adapter)])\nmanifest["upstream_sources"]=upstream_sources\nmanifest["upstream_original_columns"]=original_columns\nmanifest["acquisition_log"]=acquisition_log\nmanifest["input_filename"]=upload_meta["original_filename"]\nmanifest["input_sha256"]=upload_meta["sha256"]\nmanifest["input_provenance"]="user-supplied WHO-claimed CSV; independent authenticity not verified"\nmanifest["national_download_url"]=None\nmanifest["derived_input_note"]="Derived regional input GENERATED from the one user-uploaded national CSV, not a WHO regional export. Original source preserved in data/raw."\nwrite_processed(WORK,data,manifest,raw_files)\ninput_hash=hashlib.sha256(observations_csv(data)).hexdigest()\ninput_metadata={"schema_version":1,"observations_sha256":input_hash,"manifest":manifest}\n(WORK/"amr_colab_input.zip").write_bytes(input_bundle(data,manifest))\n(WORK/"data/processed/annual_changes.csv").write_bytes(annual_changes(data).to_csv(index=False).encode("utf-8"))\nprint(f"Prepared {len(data)} eligible region-year medians from a single uploaded file; 0 online API calls.")\n'
modeling_source = 'INDICATOR = "ECOLI_3GC_BSI"\nSTATISTIC = "derived_country_median"  # Explicit WHO national-data fallback; never chart medians.\nHOLDOUT_YEAR = settings.get("holdout_year")  # Default: latest eligible derived observation year.\nALPHAS = (0.1, 1.0, 10.0)\nMIN_RIDGE_ROWS = 12\nMIN_RIDGE_REGIONS = 3\nstudy = data[(data.indicator_code == INDICATOR) & (data.statistic_type == STATISTIC)].copy()\nstudy = study.dropna(subset=["resistance_pct"]).sort_values(["who_region", "year"])\nif study.empty:\n    raise ValueError("No observed primary target for this statistic. Revisit the import or STATISTIC.")\nfor _, group in study.groupby("who_region"):\n    for field in ["source_url", "source_revision", "definition_note"]:\n        if field in group and group[field].nunique() > 1:\n            raise ValueError(f"Source/definition changes within a regional target: {field}")\nholdout_year = int(study.year.max()) if HOLDOUT_YEAR is None else int(HOLDOUT_YEAR)\nif holdout_year != int(study.year.max()):\n    raise ValueError("Final holdout must be the latest imported year. Import a separately scoped study if excluding incomplete later years.")\nprint("Indicator:", INDICATOR, "| statistic:", STATISTIC, "| final holdout:", holdout_year)\nprint("Final projections follow each region\'s own last observed year, not the publication year.")\n\nREGIONS = ("AFR", "AMR", "EMR", "EUR", "SEAR", "WPR")\nFEATURES = ["who_region", "year", "lag_1", "lag_2", "lag_delta"]\nMODELS = ("naive", "linear", "ridge")\n\ndef history(region, cutoff):\n    g = study[(study.who_region == region) & (study.year < cutoff)].sort_values("year")\n    if g.empty:\n        return g\n    runs = g.year.diff().ne(1).cumsum()\n    return g[runs == runs.iloc[-1]]\n\ndef feature(region, target_year):\n    # No same-year or future row can enter these lag features.\n    g = study[(study.who_region == region) & (study.year < target_year)].set_index("year")\n    if target_year - 1 not in g.index or target_year - 2 not in g.index:\n        return None\n    a, b = float(g.loc[target_year - 1, "resistance_pct"]), float(g.loc[target_year - 2, "resistance_pct"])\n    return {"who_region": region, "year": target_year, "lag_1": a, "lag_2": b, "lag_delta": a - b}\n\ndef training_rows(cutoff):\n    rows = []\n    for obs in study[study.year < cutoff].itertuples():\n        f = feature(obs.who_region, int(obs.year))\n        if f is not None:\n            rows.append({**f, "target": float(obs.resistance_pct)})\n    result = pd.DataFrame(rows, columns=FEATURES + ["target"])\n    assert result.empty or result.year.max() < cutoff\n    return result\n\ndef build_ridge(alpha):\n    preprocessing = ColumnTransformer([\n        ("region", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["who_region"]),\n        ("numeric", StandardScaler(), ["year", "lag_1", "lag_2", "lag_delta"])\n    ])\n    return Pipeline([("preprocessing", preprocessing), ("ridge", Ridge(alpha=alpha))])\n\ndef fit_ridge(cutoff, alpha):\n    train = training_rows(cutoff)\n    if len(train) < MIN_RIDGE_ROWS or train.who_region.nunique() < MIN_RIDGE_REGIONS:\n        return None\n    model = build_ridge(alpha)\n    model.fit(train[FEATURES], train.target)\n    # All fitted transformations see training rows only.\n    assert train.year.max() < cutoff\n    return model\n\ndef bounded(raw):\n    raw = float(raw)\n    if not np.isfinite(raw):\n        raise ValueError("Nonfinite model output.")\n    return float(np.clip(raw, 0, 100)), bool(raw < 0 or raw > 100)\n\ndef baseline_raw(region, target_year):\n    h = history(region, target_year)\n    if len(h) < 4 or int(h.year.max()) != target_year - 1:\n        return None\n    x = h.year.to_numpy(dtype=float)\n    y = h.resistance_pct.to_numpy(dtype=float)\n    # Center year for numerical stability. This fit runs only in Colab.\n    xc = x - x.mean()\n    slope = np.dot(xc, y - y.mean()) / np.dot(xc, xc)\n    return {"naive": float(y[-1]), "linear": float(y.mean() + slope * (target_year - x.mean()))}\n\ndef tune_before(cutoff):\n    errors = {alpha: [] for alpha in ALPHAS}\n    for year in sorted(study.year.unique()):\n        year = int(year)\n        if year >= cutoff:\n            continue\n        eligible = []\n        for obs in study[study.year == year].itertuples():\n            if baseline_raw(obs.who_region, year) is not None:\n                eligible.append(obs)\n        if not eligible:\n            continue\n        for alpha in ALPHAS:\n            model = fit_ridge(year, alpha)\n            if model is None:\n                continue\n            for obs in eligible:\n                f = feature(obs.who_region, year)\n                pred, _ = bounded(model.predict(pd.DataFrame([f])[FEATURES])[0])\n                errors[alpha].append(abs(pred - float(obs.resistance_pct)))\n    if min(map(len, errors.values())) < 3:\n        return 1.0, "Default alpha=1; fewer than 3 earlier chronological validation cases."\n    alpha = min(ALPHAS, key=lambda a: (float(np.mean(errors[a])), a))\n    return alpha, f"Earlier rolling-origin MAE; {len(errors[alpha])} cases; all targets < {cutoff}."\n\n# Structural assertions use real imported rows, and do not fit extra models.\nfor obs in study.itertuples():\n    f = feature(obs.who_region, int(obs.year))\n    if f is not None:\n        assert f["year"] == obs.year\n        past = study[(study.who_region == obs.who_region) & (study.year == obs.year - 1)]\n        assert f["lag_1"] == float(past.resistance_pct.iloc[0])\nprint("Lag chronology assertions passed.")\n\nrows, exclusions, chronological_settings = [], [], []\nfrozen_alpha, frozen_reason = tune_before(holdout_year)\nfor target_year in sorted(study.year.unique()):\n    target_year = int(target_year)\n    if target_year > holdout_year:\n        continue\n    # The holdout targets never influence tuning, even for final selection.\n    alpha, reason = (frozen_alpha, frozen_reason) if target_year == holdout_year else tune_before(target_year)\n    ridge = fit_ridge(target_year, alpha)\n    chronological_settings.append({"target_year": target_year, "alpha": alpha, "reason": reason,\n                     "ridge_training_rows": len(training_rows(target_year)), "ridge_available": ridge is not None})\n    for obs in study[study.year == target_year].itertuples():\n        raw = baseline_raw(obs.who_region, target_year)\n        if raw is None:\n            exclusions.append({"who_region": obs.who_region, "target_year": target_year,\n                               "reason": "Require 4 consecutive prior observations ending at t-1."})\n            continue\n        raw["ridge"] = None\n        if ridge is not None:\n            raw["ridge"] = float(ridge.predict(pd.DataFrame([feature(obs.who_region, target_year)])[FEATURES])[0])\n        else:\n            exclusions.append({"who_region": obs.who_region, "target_year": target_year,\n                               "reason": "Ridge unavailable: insufficient pooled training rows/regions."})\n        row = {"who_region": obs.who_region, "indicator_code": INDICATOR, "statistic_type": STATISTIC,\n               "target_year": target_year, "actual": float(obs.resistance_pct),\n               "train_end_year": int(study[study.year < target_year].year.max()),\n               "phase": "holdout" if target_year == holdout_year else "validation",\n               "record_type": "backtest_prediction", "observations_sha256": input_hash}\n        for name in MODELS:\n            if raw[name] is None:\n                row[name + "_pred"], row[name + "_raw"], row[name + "_clipped"] = np.nan, np.nan, False\n            else:\n                row[name + "_pred"], row[name + "_clipped"] = bounded(raw[name])\n                row[name + "_raw"] = raw[name]\n        rows.append(row)\nbacktest_columns = ["who_region", "indicator_code", "statistic_type", "target_year", "actual",\n                    "train_end_year", "phase", "record_type", "observations_sha256"] + [\n                    name + suffix for name in MODELS for suffix in ("_pred", "_raw", "_clipped")]\nbacktests = pd.DataFrame(rows, columns=backtest_columns)\ndisplay(backtests)\nprint("Final holdout alpha:", frozen_alpha, frozen_reason)\n\ndef score(rows):\n    models = list(MODELS) if rows.ridge_pred.notna().any() else ["naive", "linear"]\n    shared = rows.dropna(subset=["actual"] + [m + "_pred" for m in models])\n    scores = []\n    if not shared.empty:\n        for model in models:\n            error = shared[model + "_pred"] - shared.actual\n            scores.append({"model": model, "mae_pp": float(error.abs().mean()),\n                           "rmse_pp": float(np.sqrt((error ** 2).mean())), "n": len(shared),\n                           "comparison": "matched_three_models" if len(models) == 3 else "matched_baselines"})\n    return pd.DataFrame(scores, columns=["model", "mae_pp", "rmse_pp", "n", "comparison"])\n\nholdout = backtests[backtests.phase == "holdout"]\nmetrics = score(holdout)\ndisplay(metrics)\nif metrics.empty or metrics.n.min() < 3:\n    interpretation = "Insufficient evaluation coverage for meaningful model ranking"\nelif "ridge" not in metrics.model.values:\n    interpretation = "Ridge unavailable; only transparent baselines could be evaluated."\nelse:\n    scores = metrics.set_index("model")\n    interpretation = ("Ridge has lower MAE than naive on identical holdout cases."\n                      if scores.loc["ridge", "mae_pp"] < scores.loc["naive", "mae_pp"]\n                      else "Ridge did not improve MAE over naive on identical holdout cases.")\nprint(interpretation)\ndisplay(pd.DataFrame(exclusions))\n\nvalidation_metrics = score(backtests[backtests.phase == "validation"])\npreferred_model = ("naive" if validation_metrics.empty or validation_metrics.n.min() < 3\n                   else validation_metrics.sort_values(["mae_pp", "model"]).iloc[0].model)\nforecast_rows = []\nartifact_dir = WORK / "models"\nartifact_dir.mkdir(exist_ok=True)\nfinal_artifacts = []\nfor region in REGIONS:\n    g = study[study.who_region == region]\n    if g.empty:\n        exclusions.append({"who_region": region, "reason": "No observations for target."})\n        continue\n    last = int(g.year.max())\n    target_year = last + 1\n    raw = baseline_raw(region, target_year)\n    if raw is None:\n        exclusions.append({"who_region": region, "target_year": target_year, "reason": "Final history has fewer than 4 consecutive years."})\n        continue\n    # Each region\'s projection uses no observations from its target year, even if\n    # another region has newer observed data in the bundle.\n    # Stale regional endpoints also require past-only hyperparameter decisions.\n    region_alpha = frozen_alpha if target_year > holdout_year else tune_before(target_year)[0]\n    ridge = fit_ridge(target_year, region_alpha)\n    raw["ridge"] = None if ridge is None else float(ridge.predict(pd.DataFrame([feature(region, target_year)])[FEATURES])[0])\n    if ridge is not None:\n        artifact_path = artifact_dir / f"ridge_{region}_through_{last}.joblib"\n        joblib.dump(ridge, artifact_path)\n        final_artifacts.append(str(artifact_path))\n    for model in MODELS:\n        if raw[model] is None:\n            continue\n        prediction, clipped = bounded(raw[model])\n        forecast_rows.append({"who_region": region, "indicator_code": INDICATOR, "statistic_type": STATISTIC,\n                              "target_year": target_year, "last_observed_year": last, "model": model,\n                              "prediction": prediction, "raw_prediction": raw[model], "clipped": clipped,\n                              "record_type": "experimental_forecast", "history_years": len(history(region, target_year)),\n                              "ridge_alpha": region_alpha if model == "ridge" else np.nan,\n                              "observations_sha256": input_hash,\n                              "label": f"Forecast for {target_year}; not observed / not independently verified",\n                              "limitation": "Exploratory / limited historical data; changing surveillance coverage."})\nforecasts = pd.DataFrame(forecast_rows, columns=["who_region", "indicator_code", "statistic_type",\n    "target_year", "last_observed_year", "model", "prediction", "raw_prediction", "clipped",\n    "record_type", "history_years", "ridge_alpha", "observations_sha256", "label", "limitation"])\ndisplay(forecasts)\nprint("Preferred method from pre-holdout validation:", preferred_model)\n\nmetadata = {\n    "schema_version": 1, "execution_environment": "google_colab",\n    "created_at": datetime.now(timezone.utc).isoformat(), "observations_sha256": input_hash,\n    "indicator_code": INDICATOR, "statistic_type": STATISTIC, "holdout_year": holdout_year,\n    "holdout_note": "Latest imported observation year; source completeness must be reviewed by user.",\n    "source_manifest": input_metadata["manifest"], "alpha": frozen_alpha,\n    "alpha_reason": frozen_reason, "alpha_candidates": list(ALPHAS),\n    "minimum_ridge_training_rows": MIN_RIDGE_ROWS, "minimum_ridge_training_regions": MIN_RIDGE_REGIONS,\n    "minimum_prior_consecutive_years": 4, "feature_names": FEATURES,\n    "ridge_seed": None, "seed_note": "Ridge is deterministic; no randomized split or estimator is used.",\n    "chronological_settings": chronological_settings, "exclusions": exclusions, "interpretation": interpretation,\n    "preferred_model": preferred_model, "selection_rule": "Minimum pre-holdout matched validation MAE with >=3 cases; otherwise naive.",\n    "final_refit": "Use frozen alpha; refit with labels strictly earlier than each region projection year.",\n    "versions": {"python": platform.python_version(), "numpy": np.__version__,\n                 "pandas": pd.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__},\n    "limitation": "Not clinical guidance. Median of reporting countries is not population prevalence. No independent future validation.",\n    "fitted_artifacts": [Path(p).name for p in final_artifacts]\n}\nresult_dir = WORK / "results"\nresult_dir.mkdir(exist_ok=True)\nbacktests.to_csv(result_dir / "backtests.csv", index=False)\nmetrics.to_csv(result_dir / "model_metrics.csv", index=False)\nforecasts.to_csv(result_dir / "forecasts.csv", index=False)\n(result_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\noutput_zip = WORK / "amr_colab_results.zip"\nwith zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:\n    for path in sorted(result_dir.iterdir()):\n        archive.write(path, path.name)\nprint("Result bundle:", output_zip)\nprint("Return amr_colab_results.zip to the local dashboard. No models need to be trained locally.")'


In [ ]:
reporting_source = '\nfrom src.results import load_results\n# Validate the produced result archive with the same contract as the local app.\nverified = load_results((WORK / "amr_colab_results.zip").read_bytes(), data)\nquality = quality_report(data)\nquality.to_csv(WORK / "data/processed/data_quality.csv", index=False)\n\nplt.rcParams.update({"figure.figsize": (10, 5), "axes.spines.top": False,\n                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .18})\npalette = {"AFR": "#087f8c", "AMR": "#c56a3c", "EMR": "#657593",\n           "EUR": "#586c50", "SEAR": "#8b5880", "WPR": "#a38b31"}\nfigures = []\ndef save_figure(fig, filename):\n    fig.tight_layout()\n    path = WORK / "reports/figures" / filename\n    fig.savefig(path, dpi=150, bbox_inches="tight")\n    plt.close(fig)\n    figures.append(path)\n\nfig, ax = plt.subplots()\nfor region in REGIONS:\n    g = data[data.who_region == region].sort_values("year")\n    if not g.empty:\n        observed = g.set_index("year").resistance_pct.reindex(range(int(g.year.min()), int(g.year.max()) + 1))\n        ax.plot(observed.index, observed.values, marker="o", color=palette[region], label=region)\nax.set(title="Observed derived country medians (gaps retained)", xlabel="Observation year", ylabel="Resistance (%)")\nax.legend(ncol=3)\nsave_figure(fig, "01_regional_trends.png")\n\nlatest_year = int(data.year.max())\nlatest = data[data.year == latest_year].set_index("who_region").reindex(REGIONS)\nfig, ax = plt.subplots()\nshown = latest.resistance_pct.dropna()\nax.bar(shown.index, shown.values, color=[palette[r] for r in shown.index])\nax.set(title=f"Same-year comparison: {latest_year} (available regions only)", ylabel="Observed derived median (%)")\nsave_figure(fig, "02_latest_comparison.png")\n\ncount_matrix = coverage.pivot(index="who_region", columns="year", values="country_count").reindex(REGIONS)\nfig, ax = plt.subplots()\nvalues = count_matrix.to_numpy(dtype=float)\nimage = ax.imshow(np.ma.masked_invalid(values), aspect="auto", cmap="YlGnBu", vmin=0)\nax.set_yticks(range(len(REGIONS)), REGIONS)\nax.set_xticks(range(len(count_matrix.columns)), [str(y) for y in count_matrix.columns], rotation=45)\nfor i in range(values.shape[0]):\n    for j in range(values.shape[1]):\n        if np.isfinite(values[i, j]):\n            ax.text(j, i, str(int(values[i, j])), ha="center", va="center", fontsize=9)\nfig.colorbar(image, ax=ax, label="Number of reporting countries")\nax.set(title="Surveillance participation (counts below 5 are suppressed in resistance analysis)", xlabel="Observation year")\nsave_figure(fig, "03_country_coverage.png")\n\nif not metrics.empty:\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    for ax, measure, title in zip(axes, ["mae_pp", "rmse_pp"], ["MAE", "RMSE"]):\n        ax.bar(metrics.model, metrics[measure], color=["#087f8c", "#c56a3c", "#657593"][:len(metrics)])\n        ax.set(title=f"Matched holdout {title} (n={int(metrics.n.min())})", ylabel="Percentage points")\n    save_figure(fig, "04_matched_evaluation.png")\n\nif not holdout.empty:\n    fig, ax = plt.subplots()\n    for model in MODELS:\n        valid = holdout.dropna(subset=[model + "_pred"])\n        if not valid.empty:\n            ax.scatter(valid.actual, valid[model + "_pred"], label=model, alpha=.8)\n    ax.plot([0, 100], [0, 100], "--", color="#657593", linewidth=1)\n    ax.set(xlabel="Observed (%)", ylabel="Backtest prediction (%)", title=f"Predicted versus actual: {holdout_year} holdout")\n    ax.legend()\n    save_figure(fig, "05_predicted_vs_actual.png")\n\nif not forecasts.empty:\n    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)\n    for region, ax in zip(REGIONS, axes.flat):\n        g = data[data.who_region == region].sort_values("year")\n        if not g.empty:\n            observed = g.set_index("year").resistance_pct.reindex(range(int(g.year.min()), int(g.year.max()) + 1))\n            ax.plot(observed.index, observed.values, "o-", color=palette[region], label="Observed")\n        for model, marker in zip(MODELS, ["D", "s", "^"]):\n            p = forecasts[(forecasts.who_region == region) & (forecasts.model == model)]\n            if not p.empty:\n                ax.scatter(p.target_year, p.prediction, marker=marker, s=65, label=model + " experimental")\n        ax.set(title=region, xlabel="Observation / target year", ylabel="Resistance (%)")\n        if g.empty:\n            ax.text(.5, .5, "Data unavailable", transform=ax.transAxes, ha="center")\n        else:\n            ax.legend(fontsize=7)\n    save_figure(fig, "06_experimental_forecasts.png")\n\neligible_holdout_regions = sorted(holdout.who_region.unique()) if not holdout.empty else []\nexcluded_holdout_regions = sorted(set(REGIONS) - set(eligible_holdout_regions))\nsources_html = "".join("<li>" + html.escape(x["filename"]) + " — " +\n    html.escape(x["source_url"]) + " — SHA-256 " + html.escape(x["sha256"]) + "</li>" for x in upstream_sources)\nimage_html = ""\nimport base64\nfor path in figures:\n    image_html += \'<figure><img style="max-width:100%" src="data:image/png;base64,\' + base64.b64encode(path.read_bytes()).decode() + \'"><figcaption>\' + html.escape(path.name) + "</figcaption></figure>"\nstyle = """<style>body{font:16px/1.6 system-ui;max-width:1120px;margin:36px auto;padding:0 20px;color:#193344;background:#f7f8f5}h1,h2{line-height:1.2}table{border-collapse:collapse;font-size:13px;display:block;overflow:auto}th,td{border:1px solid #dce4e1;padding:7px;text-align:left}aside{padding:18px;background:#edf1ef}code{word-break:break-all}figure{margin:30px 0}li{overflow-wrap:anywhere}</style>"""\nsummary = (\n    f"{len(national)} national observations from {national.country_code.nunique()} reporting countries; "\n    f"{len(data)} eligible regional medians over {int(data.year.min())}–{int(data.year.max())}. "\n    f"Holdout year: {holdout_year}. Eligible holdout regions: {\', \'.join(eligible_holdout_regions) or \'none\'}. "\n    f"Excluded holdout regions: {\', \'.join(excluded_holdout_regions) or \'none\'}.")\ncaveat = (\n    "These are derived unweighted medians of available WHO national percentages, not WHO dashboard regional statistics, "\n    "adjusted estimates, or population prevalence. Country participation changes across years. Infection/testing counts are "\n    "unknown, so the dashboard\'s count-based eligibility is not reproduced. Region assignments use a bundled WHO Member State May-2025 snapshot (Indonesia now WPR), not reconstructed historical membership. Input authenticity has not been independently verified. Missing years are retained. "\n    "Direction uses an application heuristic, not statistical significance. All future points are experimental one-year "\n    "projections, not observed values or clinical advice. No confidence bounds are fabricated.")\n# Append EDA charts to the report; this step never changes model data.\nfor path in eda_figures:\n    image_html += \'<figure><img style="max-width:100%" src="data:image/png;base64,\' + base64.b64encode(path.read_bytes()).decode() + \'"><figcaption>EDA: \' + html.escape(path.name) + \'</figcaption></figure>\'\neda_sections = ("<h2>EDA — observed national data</h2><p>No predicted values are included in these summaries. "\n                "Country and regional distributions are sensitive to reporting coverage and not representative of prevalence.</p>" +\n                by_region.to_html(index=False, escape=True) + by_year.to_html(index=False, escape=True))\n\nreport_html = (\n    "<!doctype html><meta charset=\'utf-8\'><title>AMR Sentinel analysis</title>" + style +\n    "<h1>AMR Sentinel — end-to-end analysis</h1><p>Generated " + html.escape(datetime.now(timezone.utc).isoformat()) +\n    "</p><aside>" + html.escape(caveat) + "</aside><h2>Data and source release</h2><p>" + html.escape(summary) +\n    "</p><p>Source revision: " + html.escape(revision) + ".</p><h2>Chronological methods</h2>"\n    "<p>Naïve carry-forward, linear extrapolation and pooled Ridge. Main cases require four consecutive prior years. "\n    "Ridge requires at least 12 complete lag training rows from three regions. Year, lags and one-hot region transformations "\n    "are fitted on past-only training rows. Alpha tuning uses earlier rolling origins; the holdout is untouched. "\n    "Predictions are bounded to [0,100] with raw values and clipping flags. Metrics compare identical cases. "\n    "The preferred method is chosen on pre-holdout validation; sparse validation defaults to naïve.</p>"\n    "<h2>EDA</h2>" + eda_sections + "<h2>Coverage and descriptive trends</h2>" + quality.to_html(index=False, escape=True) +\n    "<h2>Matched holdout results</h2><p>" + html.escape(interpretation) + "</p>" +\n    metrics.to_html(index=False, escape=True) + "<h2>Individual holdout errors</h2>" +\n    holdout.to_html(index=False, escape=True) + "<h2>Experimental projections</h2>" +\n    forecasts.to_html(index=False, escape=True) + "<h2>Figures</h2>" + image_html +\n    "<h2>Exclusions and settings</h2><pre>" + html.escape(json.dumps({"exclusions": exclusions, "settings": chronological_settings, "alpha": frozen_alpha, "alpha_reason": frozen_reason}, indent=2)) +\n    "</pre><h2>Sources and provenance</h2><ul>" + sources_html +\n    "</ul><p><a href=\'" + INDICATOR_PAGE + "\'>WHO E. coli indicator</a> · "\n    "<a href=\'https://worldhealthorg.shinyapps.io/glass-dashboard/\'>WHO GLASS dashboard</a> · "\n    "<a href=\'https://www.who.int/data/gho/info/gho-odata-api\'>WHO API documentation</a></p>"\n)\n(WORK / "reports/analysis_report.html").write_text(report_html, encoding="utf-8")\nreport_md = (\n    "# AMR Sentinel analysis\\n\\n" + summary + "\\n\\n" + caveat +\n    "\\n\\n## Model result\\n\\n" + interpretation +\n    "\\n\\nHoldout metrics (percentage points):\\n\\n" + metrics.to_csv(index=False) +\n    "\\n\\nDetailed tables, source provenance and embedded figures are in analysis_report.html. "\n    "Reproducible raw/processed data, settings and pinned package inventory accompany this report.\\n")\n(WORK / "reports/analysis_report.md").write_text(report_md, encoding="utf-8")\n(WORK / "reports/acquisition_log.json").write_text(json.dumps(acquisition_log, indent=2), encoding="utf-8")\nfreeze = __import__("subprocess").check_output([sys.executable, "-m", "pip", "freeze"], text=True)\n(WORK / "requirements_colab.txt").write_text(freeze, encoding="utf-8")\nprint(summary)\nprint(interpretation)\nprint("Report, charts and matched-case result validation completed.")\n'
packaging_source = '\n# Plain-data project bundle excludes fitted artifacts; models are a separate ZIP.\nproject_zip = WORK / "amr_project_bundle.zip"\nwith zipfile.ZipFile(project_zip, "w", zipfile.ZIP_DEFLATED) as archive:\n    for directory in ("data", "results", "reports"):\n        for path in sorted((WORK / directory).rglob("*")):\n            if path.is_file():\n                archive.write(path, path.relative_to(WORK).as_posix())\n    for name in ("run_settings.json", "requirements_colab.txt", "amr_colab_input.zip", "amr_colab_results.zip"):\n        archive.write(WORK / name, name)\nif final_artifacts:\n    with zipfile.ZipFile(WORK / "amr_colab_models.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n        for filename in final_artifacts:\n            archive.write(filename, Path(filename).name)\nprint("Download amr_project_bundle.zip to return data, analysis and results together.")\n'


## 7. Execute complete CSV → EDA → Machine Learning study

This cell copies the previously uploaded CSV into a unique run folder, preflights all Python stages, and runs them in the Colab Python worker. It uses **no WHO data download URLs**. All EDA results, model metrics, historical backtests, experimental forecasts and provenance reports are saved in the run directory. Errors produce diagnostic files rather than fabricated results.


In [ ]:
ANALYSIS_SUCCEEDED = False
from uuid import uuid4
# A rerun gets a new directory: stale figures and raw payloads cannot leak in.
WORK = Path("/content/amr_sentinel") / uuid4().hex
WORK.mkdir(parents=True, exist_ok=False)
if not globals().get("INPUT_CSV") or not INPUT_CSV.is_file():
    raise RuntimeError("Upload one WHO E. coli CSV in the upload cell before running the analysis.")
import shutil
shutil.copyfile(INPUT_CSV, WORK / "who_ecoli_upload_pending.csv")
(WORK / "data/raw").mkdir(parents=True, exist_ok=True)
shutil.move(str(WORK / "who_ecoli_upload_pending.csv"), str(WORK / "data/raw/who_ecoli_uploaded.csv"))
(WORK / "upload_metadata.json").write_text(json.dumps(INPUT_META,indent=2),encoding="utf-8")
(WORK / "run_settings.json").write_text(json.dumps(run_settings, indent=2), encoding="utf-8")
for relative, content in validator_assets.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
worker_body = "\n\n".join([worker_prelude, acquisition_source, processing_source,
                            eda_source, modeling_source, reporting_source, packaging_source])
# Preflight all emitted Python code before launching the worker process.
import ast
for relative, content in validator_assets.items():
    ast.parse(content, filename=relative)
for title, content in [("worker prelude", worker_prelude), ("acquisition", acquisition_source),
                       ("processing", processing_source), ("EDA", eda_source), ("modeling", modeling_source),
                       ("reporting", reporting_source), ("packaging", packaging_source)]:
    ast.parse(content, filename=title)
print("Preflight passed: all generated modules and worker stages parse successfully.")
# Save diagnostic metadata and a complete traceback if acquisition or analysis fails.
diagnostic_wrapper = """
if __name__ == "__main__":
    try:
        exec(compile(WORKER_BODY, "<AMR Sentinel Colab worker>", "exec"))
    except Exception:
        import traceback, json
        from pathlib import Path
        from datetime import datetime, timezone
        work = Path(__file__).resolve().parent
        diagnostic = {"failed_at": datetime.now(timezone.utc).isoformat(),
                      "traceback": traceback.format_exc(),
                      "acquisition_log": globals().get("acquisition_log", []),
                      "upstream_sources": globals().get("upstream_sources", [])}
        (work / "failure_diagnostics.json").write_text(json.dumps(diagnostic, indent=2))
        raise
"""
script = WORK / "analysis_worker.py"
script.write_text("WORKER_BODY = " + repr(worker_body) + "\n" + diagnostic_wrapper, encoding="utf-8")
# Reuse worker_env from setup: it may contain the verified target PYTHONPATH.
analysis_env = worker_env.copy()
analysis_env["AMR_COLAB_AUTHORIZED"] = "1"
with (WORK / "run.log").open("w", encoding="utf-8") as logfile:
    process = subprocess.Popen([str(worker_python), "-u", str(script)], cwd=WORK,
                               env=analysis_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        logfile.write(line)
        logfile.flush()
    returncode = process.wait()
if returncode:
    print(f"Run failed. Download {WORK / 'failure_diagnostics.json'} and {WORK / 'run.log'} from Colab's Files panel.")
    raise RuntimeError(f"Analysis worker failed (exit {returncode}); see the full output above. No results from this run are verified.")
ANALYSIS_SUCCEEDED = True
print("Completed: uploaded CSV → validation → EDA → ML evaluation → reports → exports (no WHO API).")

## 8. Review EDA, model evaluation and the final report

The HTML report includes summary statistics, data-quality tables, observed-only EDA charts, chronological model error metrics and forecast plots. The report distinguishes observed values from experimental forecasts; missing data remain missing.


In [ ]:
if not globals().get("ANALYSIS_SUCCEEDED"):
    raise RuntimeError("Run the analysis cell successfully before displaying any outputs.")
from IPython.display import display, HTML, Image
report_path = WORK / "reports/analysis_report.html"
display(HTML(report_path.read_text(encoding="utf-8")))
print("EDA charts and tables are also included in the HTML report and ZIP.")
for figure in sorted((WORK / "reports/eda").glob("*.png")):
    display(Image(filename=str(figure)))
for figure in sorted((WORK / "reports/figures").glob("*.png")):
    display(Image(filename=str(figure)))

## 9. Download the results ZIP

`amr_project_bundle.zip` includes the **original uploaded CSV**, hash/provenance information, national and regional processed CSVs, EDA tables/PNGs, chronological backtests, model results and the HTML report. Optional separate model ZIP contains fitted `.joblib` files. Do **not** treat the uploaded file as WHO-verified unless you obtained it from the cited WHO indicator page.


In [ ]:
if not globals().get("ANALYSIS_SUCCEEDED"):
    raise RuntimeError("No verified study bundle is available; the analysis must succeed first.")
files.download(str(WORK / "amr_project_bundle.zip"))
DOWNLOAD_SMALL_RESULTS_SEPARATELY = False
DOWNLOAD_FITTED_PIPELINES = False
if DOWNLOAD_SMALL_RESULTS_SEPARATELY:
    files.download(str(WORK / "amr_colab_results.zip"))
if DOWNLOAD_FITTED_PIPELINES and (WORK / "amr_colab_models.zip").exists():
    files.download(str(WORK / "amr_colab_models.zip"))
print("Send amr_project_bundle.zip back. No local training is required.")

## Troubleshooting and data rules

- **WHO HTTP 500:** no longer applicable; this notebook does not call WHO data APIs. If it does, you opened an older notebook.
- **No browser file dialog:** run the upload cell manually in Google Colab, select one real CSV, and then use **Runtime → Run all** (which prompts again).
- **Unknown country names/codes:** inspect `data/processed/input_validation.json`; the bundled lookup covers 194 WHO Member States plus three recognized additional areas, with May-2025 region membership. Unknown values are excluded and logged. CSV rows with no recognized country identifier cannot be analyzed.
- **Missing resistance or year headers:** inspect the header printed after upload; rename columns to recognized aliases in a copy of your CSV.
- **Multiple country/year records:** unresolved stratification is not averaged; resolve WHO export dimensions before rerunning.
- **Too few countries:** minimum of five reporting countries per WHO region/year; groups below this threshold cannot enter regional ML training.
- **Insufficient historical coverage:** some or all Ridge forecasts may be unavailable. This is not a reason to fabricate a result.
- **Missing EDA figures:** only data-dependent charts are produced; EDA CSV tables are always written after successful preprocessing.
- **Colab package conflicts:** reset runtime; this notebook does not create `venv` or call `ensurepip`.
- **Failures:** inspect `/content/amr_sentinel/<run_id>/failure_diagnostics.json` and `run.log`; source metadata saved inside each run folder.

### Source and methodological attribution
WHO indicator: https://data.who.int/indicators/i/918081E/745F475

WHO geographical grouping: https://www.who.int/countries

WHO six-region reference: https://www.who.int/about/who-we-are/regional-offices

WHO Member State classifications (194 countries; updated through May 2025): https://shanlong-who.github.io/DSIR/reference/who_countries.html

**Important:** Yearly unweighted medians of available national percentages are *derived by this notebook*. They are not adjusted WHO GLASS regional surveillance estimates. Model outputs are exploratory only, not clinical decision support.
